# DataSpeak — Time Series & Forecasting Agent
## Production Build v3.0 — Complete Notebook

**Role**: Emad — Time Series Specialist  
**LLM**: Groq / llama-3.3-70b-versatile  
**Team**: Farhat (Cleaning Agent) · Motawea (EDA/Visualization Agent)

---

### Pipeline
```
CSV → SchemaAdapter → validate_dataframe → TimeSeriesEngine
    → TemplateRenderer → TimeSeriesAgent (LLM) → Final Report
```

### All Stage-1 Bugs Fixed
| # | Bug | Fix |
|---|-----|-----|
| 1 | Mutable THRESHOLDS default arg | None-guard + `_get_thresholds()` deep-merger |
| 2 | Seasonality misleads on short data | `reliable` flag gates seasonal correction |
| 3 | Adaptive threshold used raw CV | Residual CV via `_adaptive_threshold_from_residuals()` |
| 4 | Forecast 4× magnitude error | `x_f = last_x + (w / 4.33)` fractional months |
| 5 | pct_change hid adjusted baseline | `pct_change_baseline_month` key exposed |
| 6 | Equal-revenue table double-labels | `if equal: tag = ""` guard |
| 7 | `_should_alert` via internal call | Surfaced on `report["should_alert"]` |
| 8 | Global `warnings.filterwarnings` | Scoped context manager |

---

### Quick Start
1. Set API key: `export GROQ_API_KEY="gsk_..."`
2. Place `cleaned_data_1.csv` and `cleaned_data_2.csv` in the same directory
3. Run all cells top to bottom
4. **No CSV?** Jump to the **Demo** section — synthetic data is generated automatically


## 0. Install Dependencies
Run once per environment — safe to skip if already installed.

In [ ]:
# Install all required packages
# Uncomment and run if you haven't installed them yet
# !pip install pandas numpy scipy statsmodels langchain langchain-groq plotly
print('If you see no errors below, all imports succeeded.')


## 1. Imports & Logger Setup


In [ ]:
# ── Standard library ───────────────────────────────────────────────────────────
import contextlib
import json
import logging
import os
import warnings
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

# ── Third-party ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.stattools import adfuller

# ── LangChain / Groq ───────────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage


## 2. Logger & Global Constants
Configures named logger `dataspeak.ts_agent`, month/weekday name lists, and the `_DEFAULT_THRESHOLDS` sensitivity dict.  
> **Never mutate `_DEFAULT_THRESHOLDS` directly** — use `_get_thresholds(override)` to get a deep-merged copy.


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)-8s]  %(name)s  —  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("dataspeak.ts_agent")


MONTHS: List[str] = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]
WEEKDAYS: List[str] = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# ---------------------------------------------------------------------------
# All sensitivity knobs live here.
# Pass a custom copy to TimeSeriesEngine / TemplateRenderer / TimeSeriesAgent
# if you need per-store or per-experiment overrides — never mutate this dict.
# ---------------------------------------------------------------------------
_DEFAULT_THRESHOLDS: Dict[str, Any] = {
    "trend": {
        "reliable_r2":    0.60,
        "moderate_r2":    0.30,
        "significant_p":  0.05,
        "strong_growth":  15.0,   # % total change
        "mild_growth":     5.0,
        "mild_decline":   -5.0,
        "strong_decline": -15.0,
    },
    "anomaly": {
        "base_zscore":  2.5,
        "cv_high":      0.60,   # coefficient of variation breakpoints
        "cv_medium":    0.35,
        "cv_low":       0.15,
        "z_high_cv":    3.5,    # threshold for very volatile series
        "z_medium_cv":  3.0,
        "z_low_cv":     2.5,
        "z_stable_cv":  2.0,
    },
    "seasonality": {
        "strong_peak": 1.50,
        "mild_peak":   1.20,
        "strong_low":  0.70,
        "min_months_reliable": 6,   # indices below this count are informative only
    },
    "forecast": {
        "confidence_z":           1.28,   # 80 % CI
        "margin_growth_per_week": 0.04,   # widen CI 4 % per additional week
        "weeks_per_month":        4.33,   # used to convert weekly steps → month fractions
    },
    "validation": {
        "min_calendar_months": 3,
        "min_days_span":       60,
        "recommended_months":  6,
        "min_adf_months":      6,   # skip ADF test below this many monthly points
    },
}


def _get_thresholds(override: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    Return a deep-merged threshold dict.

    If *override* is None the module-level defaults are returned as-is
    (read-only reference — callers must not mutate it).  Passing a partial
    dict merges only the provided keys, leaving everything else at defaults.
    """
    if override is None:
        return _DEFAULT_THRESHOLDS
    import copy
    merged = copy.deepcopy(_DEFAULT_THRESHOLDS)
    for section, values in override.items():
        if isinstance(values, dict) and section in merged:
            merged[section].update(values)
        else:
            merged[section] = values
    return merged


## 3. SchemaAdapter — Column Inference & Normalisation
Converts any real-world CSV schema to `(order_date, revenue)`.  
**For Farhat:** your CSV can have any column names — fuzzy matching handles it.  
Negative revenue rows (refunds) are **isolated and returned separately** — do NOT remove them.


In [ ]:
class SchemaAdapter:
    """
    Convert an arbitrary real-world CSV schema into the canonical
    ``(order_date: datetime64[ns], revenue: float64)`` schema expected
    by ``TimeSeriesEngine``.

    Column inference is case-insensitive and supports fuzzy name lists.
    Currency symbols, comma separators, and trailing whitespace are
    stripped from the revenue column automatically.

    ── For Farhat ──────────────────────────────────────────────────────────────
    Your cleaned CSV can have any column names — as long as one of the date
    candidates and one of the revenue candidates below is present, the adapter
    will find it.  Negative-revenue rows (refunds / chargebacks) are isolated
    and returned separately; do NOT remove them in your cleaning step.
    """

    DATE_CANDIDATES: Tuple[str, ...] = (
        "order_date", "transaction_date", "date", "timestamp",
        "purchase_date", "datetime", "invoice_date", "sale_date",
        "created_at", "order_created",
    )

    REVENUE_CANDIDATES: Tuple[str, ...] = (
        "revenue", "sales", "amount", "total_spent", "total",
        "price", "income", "gross_sales", "net_sales", "gmv",
        "total_amount", "order_value", "sale_amount",
    )

    @classmethod
    def standardize(
        cls,
        df: pd.DataFrame,
        source_label: str = "",
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Normalise *df* and return ``(clean_df, returns_df)``.

        ``clean_df``  — non-negative revenue rows, sorted ascending, NaN-free.
        ``returns_df`` — negative-revenue rows (refunds/chargebacks); may be empty.

        Parameters
        ----------
        df:
            Raw DataFrame as loaded from CSV.
        source_label:
            Optional filename / tag used in log messages.

        Raises
        ------
        ValueError
            If no date or revenue column can be inferred.
        """
        tag = f"[{source_label}] " if source_label else ""
        work = df.copy()

        logger.info("%sSchemaAdapter — raw shape: %s", tag, work.shape)

        # ── Normalise column names ─────────────────────────────────────────
        work.columns = (
            work.columns
            .str.strip()
            .str.lower()
            .str.replace(r"[\s\-]+", "_", regex=True)
        )

        # ── Infer columns ─────────────────────────────────────────────────
        date_col    = cls._find_column(work.columns, cls.DATE_CANDIDATES)
        revenue_col = cls._find_column(work.columns, cls.REVENUE_CANDIDATES)

        if date_col is None:
            raise ValueError(
                f"{tag}Cannot infer a DATE column from: {list(work.columns)}.\n"
                f"Rename your date column to one of: {cls.DATE_CANDIDATES}"
            )
        if revenue_col is None:
            raise ValueError(
                f"{tag}Cannot infer a REVENUE column from: {list(work.columns)}.\n"
                f"Rename your revenue column to one of: {cls.REVENUE_CANDIDATES}"
            )

        logger.info("%sDate column → '%s'  |  Revenue column → '%s'", tag, date_col, revenue_col)

        # ── Canonical rename ───────────────────────────────────────────────
        work = work.rename(columns={date_col: "order_date", revenue_col: "revenue"})
        work = work[["order_date", "revenue"]]

        # ── Parse dates ────────────────────────────────────────────────────
        work["order_date"] = pd.to_datetime(work["order_date"], errors="coerce")
        bad_dates = int(work["order_date"].isna().sum())
        if bad_dates:
            logger.warning("%s%d rows dropped — unparseable date values.", tag, bad_dates)
            print(f"  ⚠  {tag}{bad_dates} rows dropped — unparseable dates.")
        work = work.dropna(subset=["order_date"])

        # ── Parse revenue (strip symbols / accounting parentheses) ─────────
        work["revenue"] = (
            work["revenue"]
            .astype(str)
            .str.replace(r"[$£€¥,\s]", "", regex=True)
            .str.replace(r"[()]", "-", regex=True)
        )
        work["revenue"] = pd.to_numeric(work["revenue"], errors="coerce")

        bad_rev = int(work["revenue"].isna().sum())
        if bad_rev:
            logger.warning("%s%d rows dropped — unparseable revenue values.", tag, bad_rev)
            print(f"  ⚠  {tag}{bad_rev} rows dropped — unparseable revenue.")
        work = work.dropna(subset=["revenue"])

        # ── Isolate negative revenue (refunds / chargebacks) ──────────────
        neg_mask   = work["revenue"] < 0
        returns_df = work[neg_mask].copy()
        clean_df   = work[~neg_mask].copy()

        if not returns_df.empty:
            total_neg = abs(float(returns_df["revenue"].sum()))
            logger.info(
                "%s%d negative-revenue rows isolated ($%.2f in returns).",
                tag, len(returns_df), total_neg,
            )
            print(
                f"  ⚠  {tag}{len(returns_df)} rows with negative revenue isolated "
                f"(${total_neg:,.2f} in returns). Excluded from engine."
            )

        # ── Sort & reset ───────────────────────────────────────────────────
        clean_df = clean_df.sort_values("order_date").reset_index(drop=True)

        logger.info(
            "%sAdapter complete — clean rows: %d  |  return rows: %d",
            tag, len(clean_df), len(returns_df),
        )
        return clean_df, returns_df

    @staticmethod
    def _find_column(
        columns: pd.Index,
        candidates: Tuple[str, ...],
    ) -> Optional[str]:
        """Return the first candidate that exists in *columns*, else None."""
        col_set = set(columns)
        for cand in candidates:
            if cand in col_set:
                return cand
        return None


## 4. TimeSeriesEngine — Pure Math Layer
Black-box mathematical engine — zero language, zero templates.  
Returns a nested facts dict including four ready-to-plot DataFrames for Motawea:
- `series["daily"]` — D-frequency
- `series["weekly"]` — W-frequency  
- `series["monthly"]` — ME-frequency  
- `series["rolling"]` — 7d/30d moving averages + confidence bands


In [ ]:
class TimeSeriesEngine:
    """
    Black-box mathematical engine.

    Input : ``clean_df`` with columns ``order_date``, ``revenue``.
    Output: nested dict of numerical facts consumed by ``TemplateRenderer``
            and exposed to ``TimeSeriesAgent``.

    ── For Motawea (Visualization Agent) ───────────────────────────────────────
    The ``run()`` method returns a dict whose ``"series"`` key contains four
    ready-to-plot DataFrames: ``daily``, ``weekly``, ``monthly``, ``rolling``.
    All have a DatetimeIndex.  Anomaly periods and forecast predictions are
    also structured for direct chart annotation — see docstring on ``run()``.
    """

    def __init__(self, thresholds: Optional[Dict[str, Any]] = None) -> None:
        # ── BUG-FIX: mutable default arg replaced with None-guard ──────────
        self.t = _get_thresholds(thresholds)

    # ── public entry point ─────────────────────────────────────────────────

    def run(self, df: pd.DataFrame) -> Dict[str, Any]:
        """
        Execute the full pipeline on *df* and return a facts dictionary.

        Return structure
        ----------------
        {
            "metadata":    { total_revenue, avg_daily_rev, ... },
            "trend":       { slope_per_month, r_squared, direction, ... },
            "seasonality": { monthly_indices, weekday_indices, reliable, ... },
            "anomalies":   { anomaly_details: [...], revenue_impact, ... },
            "forecast":    { predictions: [...], reliability, ... },
            "advanced":    { revenue_concentration_pct, yearly_peak_months, ... },
            "series": {
                "daily":   pd.DataFrame  ← for Motawea: D-frequency plot data
                "weekly":  pd.DataFrame  ← W-frequency plot data
                "monthly": pd.DataFrame  ← ME-frequency plot data
                "rolling": pd.DataFrame  ← 7d / 30d averages + confidence bands
            }
        }
        """
        ts      = self._prepare(df)
        daily   = self._resample(ts, "D")
        weekly  = self._resample(ts, "W")
        monthly = self._resample(ts, "ME")
        rolling = self._rolling(daily)

        trend       = self._trend(monthly)
        seasonality = self._seasonality(ts)
        anomalies   = self._anomalies(monthly)
        forecast    = self._forecast(monthly, trend, seasonality)
        advanced    = self._advanced(daily, ts)
        metadata    = self._metadata(df, ts, monthly)

        logger.info(
            "Engine complete — direction: %s  |  anomalies: %d  |  reliability: %s",
            trend["direction"],
            anomalies["total_detected"],
            forecast["reliability"],
        )

        return {
            "metadata":    metadata,
            "trend":       trend,
            "seasonality": seasonality,
            "anomalies":   anomalies,
            "forecast":    forecast,
            "advanced":    advanced,
            "series": {
                "daily":   daily,
                "weekly":  weekly,
                "monthly": monthly,
                "rolling": rolling,
            },
        }

    # ── preparation ────────────────────────────────────────────────────────

    def _prepare(self, df: pd.DataFrame) -> pd.DataFrame:
        """Cast types, sort, and set DatetimeIndex."""
        ts = df.copy()
        ts["order_date"] = pd.to_datetime(ts["order_date"], errors="coerce")
        ts = ts.dropna(subset=["order_date"])
        ts["revenue"] = pd.to_numeric(ts["revenue"], errors="coerce").fillna(0.0)
        ts = ts[ts["revenue"] >= 0]   # belt-and-suspenders guard
        return ts.set_index("order_date").sort_index()

    # ── resampling ─────────────────────────────────────────────────────────

    def _resample(self, ts: pd.DataFrame, rule: str) -> pd.DataFrame:
        """Aggregate to the requested frequency; mark zero-order gaps."""
        agg = ts.resample(rule).agg(
            revenue=("revenue", "sum"),
            order_count=("revenue", "count"),
        ).fillna(0)
        agg["is_gap"] = agg["order_count"] == 0
        return agg

    # ── rolling averages ───────────────────────────────────────────────────

    def _rolling(self, daily: pd.DataFrame) -> pd.DataFrame:
        """
        Compute 7-day and 30-day rolling averages with ±1σ confidence bands.

        ``min_periods=1`` ensures no NaN at the start of a short series.
        The effective window shrinks automatically if the series is shorter
        than the target window — no crash, no silent NaN flood.
        """
        r = daily[["revenue"]].copy()
        n = len(r)

        for window, label in [(7, "7d"), (30, "30d")]:
            effective = min(window, n)
            r[f"rev_{label}"] = r["revenue"].rolling(effective, min_periods=1).mean()
            std = r["revenue"].rolling(effective, min_periods=1).std().fillna(0.0)
            r[f"upper_{label}"] = r[f"rev_{label}"] + std
            r[f"lower_{label}"] = (r[f"rev_{label}"] - std).clip(lower=0.0)

        logger.debug("Rolling computed — effective windows: 7d=%d, 30d=%d", min(7, n), min(30, n))
        return r

    # ── trend ──────────────────────────────────────────────────────────────

    def _trend(self, monthly: pd.DataFrame) -> Dict[str, Any]:
        """
        Linear-regression trend on monthly revenue.

        Stationarity is tested with the Augmented Dickey-Fuller test.
        If non-stationary, regression runs on first differences to avoid a
        spurious slope; the intercept is then re-derived in level space so
        that ``_forecast`` and ``_anomalies`` can use it safely.

        The ADF test is skipped gracefully when fewer than
        ``validation.min_adf_months`` monthly points are available.
        """
        rev = monthly["revenue"].values
        n   = len(rev)
        x   = np.arange(n, dtype=float)

        min_months = self.t["validation"]["min_calendar_months"]
        if n < min_months:
            logger.warning("Trend: only %d month(s) — minimum %d required.", n, min_months)
            return self._insufficient_trend(n, min_months, rev)

        # ── stationarity test ──────────────────────────────────────────────
        adf_min = self.t["validation"]["min_adf_months"]
        is_stat: bool
        adf_p:   float

        if n < adf_min:
            is_stat = False
            adf_p   = 1.0
            logger.info("ADF skipped — only %d months (min %d).", n, adf_min)
        else:
            with _suppress_statsmodels_warnings():
                try:
                    adf_p   = float(adfuller(rev, autolag="AIC")[1])
                    is_stat = bool(adf_p < self.t["trend"]["significant_p"])
                except Exception as exc:
                    logger.warning("ADF test failed (%s); defaulting to non-stationary.", exc)
                    is_stat = False
                    adf_p   = 1.0

        # ── regression ────────────────────────────────────────────────────
        if is_stat:
            y_reg, x_reg = rev, x
        else:
            y_reg, x_reg = np.diff(rev), x[1:]

        sl, ic_diff, r_val, p_val, _ = stats.linregress(x_reg, y_reg)

        # Re-derive level-space intercept for use by anomaly / forecast layers
        ic_level = float(np.mean(rev) - sl * np.mean(x))

        sig = bool(p_val < self.t["trend"]["significant_p"])
        direction = (
            "stable"     if not sig else
            "increasing" if sl > 0  else
            "decreasing"
        )

        pct, pct_baseline_month = self._safe_pct_change(rev)
        r2 = float(r_val ** 2)

        logger.info(
            "Trend: direction=%s  slope=%.2f/mo  R²=%.4f  p=%.4f",
            direction, sl, r2, p_val,
        )

        return {
            "slope_per_month":        round(float(sl),       2),
            "slope_per_day":          round(float(sl / 30),   2),
            "intercept_diff":         round(float(ic_diff),   2),
            "intercept_level":        round(ic_level,          2),
            "r_squared":              round(r2,                4),
            "p_value":                round(float(p_val),      6),
            "adf_p_value":            round(adf_p,             6),
            "is_significant":         sig,
            "is_stationary":          is_stat,
            "direction":              direction,
            "pct_change_total":       round(float(pct),        2),
            "pct_change_baseline_month": int(pct_baseline_month),  # BUG-FIX: track adjusted baseline
            "first_month_revenue":    round(float(rev[0]),     2),
            "last_month_revenue":     round(float(rev[-1]),    2),
            "peak_month_revenue":     round(float(rev.max()),  2),
            "lowest_month_revenue":   round(float(rev.min()),  2),
            "avg_monthly_revenue":    round(float(rev.mean()), 2),
            "data_months":            n,
            "_warning":               None,
        }

    @staticmethod
    def _insufficient_trend(
        n: int,
        minimum: int,
        rev: np.ndarray,
    ) -> Dict[str, Any]:
        """Return a safe stub when too few monthly points are available."""
        mean_rev = float(rev.mean()) if len(rev) else 0.0
        return {
            "slope_per_month":        0.0,
            "slope_per_day":          0.0,
            "intercept_diff":         mean_rev,
            "intercept_level":        mean_rev,
            "r_squared":              0.0,
            "p_value":                1.0,
            "adf_p_value":            1.0,
            "is_significant":         False,
            "is_stationary":          False,
            "direction":              "insufficient_data",
            "pct_change_total":       0.0,
            "pct_change_baseline_month": 0,
            "first_month_revenue":    round(float(rev[0]),    2) if len(rev) else 0.0,
            "last_month_revenue":     round(float(rev[-1]),   2) if len(rev) else 0.0,
            "peak_month_revenue":     round(float(rev.max()), 2) if len(rev) else 0.0,
            "lowest_month_revenue":   round(float(rev.min()), 2) if len(rev) else 0.0,
            "avg_monthly_revenue":    round(mean_rev,          2),
            "data_months":            n,
            "_warning": (
                f"Only {n} calendar month(s) of data — "
                f"minimum {minimum} required for reliable trend analysis."
            ),
        }

    @staticmethod
    def _safe_pct_change(rev: np.ndarray) -> Tuple[float, int]:
        """
        Percentage change from first to last period.

        If the first period is zero the first non-zero month is used as
        the baseline.  Returns ``(pct_change, baseline_month_index)`` so
        that callers can note when the baseline was adjusted.
        """
        if len(rev) < 2:
            return 0.0, 0
        first     = rev[0]
        baseline_i = 0
        if first == 0.0:
            nonzero = np.where(rev > 0)[0]
            if len(nonzero) == 0:
                return 0.0, 0
            baseline_i = int(nonzero[0])
            first = rev[baseline_i]
        pct = ((rev[-1] - first) / first) * 100.0
        return float(pct), baseline_i

    # ── seasonality ────────────────────────────────────────────────────────

    def _seasonality(self, ts: pd.DataFrame) -> Dict[str, Any]:
        """
        Compute monthly and weekday seasonality indices.

        Index = 1.0 means exactly the observed average.

        BUG-FIX: a ``reliable`` flag is now returned.  When fewer than
        ``seasonality.min_months_reliable`` distinct months are present,
        the indices are informative but should not be used to scale a forecast.
        """
        # Monthly
        m_avg  = ts.groupby(ts.index.month)["revenue"].mean()
        m_base = float(m_avg.mean())
        if m_base == 0:
            m_idx = {MONTHS[m - 1]: 1.0 for m in m_avg.index}
        else:
            m_idx = {
                MONTHS[m - 1]: round(float(v / m_base), 4)
                for m, v in m_avg.items()
            }

        # Weekday
        w_avg  = ts.groupby(ts.index.dayofweek)["revenue"].mean()
        w_base = float(w_avg.mean())
        if w_base == 0:
            w_idx = {WEEKDAYS[i]: 1.0 for i in w_avg.index}
        else:
            w_idx = {
                WEEKDAYS[i]: round(float(v / w_base), 4)
                for i, v in w_avg.items()
            }

        pk_m = max(m_idx, key=m_idx.get)   # type: ignore[arg-type]
        lw_m = min(m_idx, key=m_idx.get)   # type: ignore[arg-type]
        pk_w = max(w_idx, key=w_idx.get)   # type: ignore[arg-type]
        lw_w = min(w_idx, key=w_idx.get)   # type: ignore[arg-type]

        min_reliable = self.t["seasonality"]["min_months_reliable"]
        reliable     = len(m_idx) >= min_reliable

        return {
            "monthly_indices":    m_idx,
            "weekday_indices":    w_idx,
            "peak_month":         pk_m,
            "low_month":          lw_m,
            "peak_month_index":   m_idx[pk_m],
            "low_month_index":    m_idx[lw_m],
            "peak_weekday":       pk_w,
            "low_weekday":        lw_w,
            "peak_weekday_index": w_idx[pk_w],
            "low_weekday_index":  w_idx[lw_w],
            "reliable":           reliable,        # BUG-FIX: new flag
            "distinct_months":    len(m_idx),
        }

    # ── anomalies ──────────────────────────────────────────────────────────

    def _anomalies(self, monthly: pd.DataFrame) -> Dict[str, Any]:
        """
        Detect anomalies via Z-score on the residuals of a level-space
        linear regression.

        BUG-FIX: the adaptive threshold is now derived from the CV of the
        *residuals*, not from the CV of raw revenue levels.  Using raw-level
        CV overstates volatility on trending series and forces a looser
        threshold exactly when tighter detection is warranted.
        """
        rev = monthly["revenue"].values
        n   = len(rev)
        x   = np.arange(n, dtype=float)

        if n < 2:
            return self._empty_anomalies(self.t["anomaly"]["base_zscore"])

        sl_lv, ic_lv, _, _, _ = stats.linregress(x, rev)
        pred      = sl_lv * x + ic_lv
        residuals = rev - pred

        r_std = float(residuals.std())
        if r_std == 0.0:
            return self._empty_anomalies(self.t["anomaly"]["base_zscore"])

        # ── BUG-FIX: use residual CV, not raw-revenue CV ──────────────────
        threshold = self._adaptive_threshold_from_residuals(residuals, rev)

        zscores = (residuals - residuals.mean()) / r_std

        details: List[Dict[str, Any]] = []
        impact  = 0.0

        for i, z in enumerate(zscores):
            if abs(z) < threshold:
                continue
            actual   = float(rev[i])
            expected = float(pred[i])
            impact  += abs(actual - expected)
            pct_dev  = (
                ((actual - expected) / expected) * 100
                if expected != 0 else 0.0
            )
            details.append({
                "period":           str(monthly.index[i].to_period("M")),
                "actual_revenue":   round(actual,   2),
                "expected_revenue": round(expected, 2),
                "zscore":           round(float(z), 4),
                "type":             "spike" if z > 0 else "drop",
                "pct_deviation":    round(pct_dev,  1),
                "severity":         "critical" if abs(z) >= 4.0 else "high",
            })

        logger.info(
            "Anomaly detection complete — threshold=%.2fσ  detected=%d",
            threshold, len(details),
        )

        return {
            "method":          "zscore_on_level_residuals",
            "threshold_used":  round(threshold, 2),
            "total_detected":  len(details),
            "spike_count":     sum(1 for d in details if d["type"] == "spike"),
            "drop_count":      sum(1 for d in details if d["type"] == "drop"),
            "revenue_impact":  round(impact, 2),
            "anomaly_details": details,
            "anomaly_periods": [d["period"] for d in details],
        }

    def _adaptive_threshold_from_residuals(
        self,
        residuals: np.ndarray,
        rev: np.ndarray,
    ) -> float:
        """
        Compute the Z-score threshold from CV of *residuals*.

        BUG-FIX vs Stage-1: CV computed on residuals (not raw revenue)
        so that trending series get an appropriately tight threshold.
        The denominator uses the mean of absolute revenue to stay positive.
        """
        mean_abs = float(np.abs(rev).mean())
        if mean_abs == 0.0:
            return self.t["anomaly"]["base_zscore"]
        cv = float(residuals.std() / mean_abs)
        at = self.t["anomaly"]
        if cv > at["cv_high"]:
            return at["z_high_cv"]
        if cv > at["cv_medium"]:
            return at["z_medium_cv"]
        if cv > at["cv_low"]:
            return at["z_low_cv"]
        return at["z_stable_cv"]

    @staticmethod
    def _empty_anomalies(threshold: float) -> Dict[str, Any]:
        return {
            "method":          "zscore_on_level_residuals",
            "threshold_used":  threshold,
            "total_detected":  0,
            "spike_count":     0,
            "drop_count":      0,
            "revenue_impact":  0.0,
            "anomaly_details": [],
            "anomaly_periods": [],
        }

    # ── forecast ───────────────────────────────────────────────────────────

    def _forecast(
        self,
        monthly:  pd.DataFrame,
        trend:    Dict[str, Any],
        season:   Dict[str, Any],
    ) -> Dict[str, Any]:
        """
        4-week-ahead forecast using::

            base      = slope × future_x  +  level_intercept
            corrected = max(base × seasonality_index, 0)   # only if season reliable
            CI        = corrected ± z80 × residual_std × (1 + growth_per_week × w)

        BUG-FIX: ``x_f`` now advances by ``w / weeks_per_month`` fractional
        months instead of the Stage-1 integer-week step.  The old approach
        applied a full month's slope per week (4× magnitude error).

        Falls back to historical mean × season index when R² < moderate_r2.
        Seasonality correction is suppressed when fewer than 6 months of data
        are present (``season["reliable"] == False``).
        """
        rev    = monthly["revenue"].values
        x      = np.arange(len(rev), dtype=float)
        sl     = trend["slope_per_month"]
        ic     = trend["intercept_level"]
        r2     = trend["r_squared"]
        m_idx  = season["monthly_indices"]
        s_ok   = season["reliable"]   # only use seasonal correction if reliable

        pred_hist = sl * x + ic
        res_std   = float(np.std(rev - pred_hist))
        mean_rev  = float(rev.mean())

        z80  = self.t["forecast"]["confidence_z"]
        grow = self.t["forecast"]["margin_growth_per_week"]
        wpm  = self.t["forecast"]["weeks_per_month"]   # 4.33

        last_x    = float(len(rev) - 1)
        last_date = monthly.index[-1]

        predictions: List[Dict[str, Any]] = []
        for w in range(1, 5):
            fd    = last_date + timedelta(days=7 * w)
            # BUG-FIX: fractional month step, not integer week step
            x_f   = last_x + (w / wpm)
            s_i   = m_idx.get(fd.strftime("%b"), 1.0) if s_ok else 1.0

            if r2 >= self.t["trend"]["moderate_r2"]:
                base = (sl * x_f + ic) * s_i
            else:
                base = mean_rev * s_i

            corr = max(float(base), 0.0)
            marg = z80 * res_std * (1.0 + grow * w)

            predictions.append({
                "date":              str(fd.date()),
                "week_number":       w,
                "predicted_revenue": round(corr,                    2),
                "lower_bound":       round(max(corr - marg, 0.0),   2),
                "upper_bound":       round(corr + marg,              2),
                "seasonality_index": round(float(s_i),               4),
                "season_applied":    s_ok,
                "method_used": (
                    "lr_x_season" if r2 >= self.t["trend"]["moderate_r2"]
                    else "mean_x_season"
                ),
            })

        reliability = (
            "high"   if r2 >= self.t["trend"]["reliable_r2"] else
            "medium" if r2 >= self.t["trend"]["moderate_r2"] else
            "low"
        )

        return {
            "method":              "linear_regression_x_seasonality_index",
            "horizon_days":        28,
            "next_week_estimate":  predictions[0]["predicted_revenue"],
            "next_month_estimate": round(
                sum(p["predicted_revenue"] for p in predictions), 2
            ),
            "confidence_level":    0.80,
            "reliability":         reliability,
            "r_squared_basis":     r2,
            "seasonality_applied": s_ok,
            "predictions":         predictions,
        }

    # ── advanced analytics ─────────────────────────────────────────────────

    def _advanced(
        self,
        daily: pd.DataFrame,
        ts:    pd.DataFrame,
    ) -> Dict[str, Any]:
        """Revenue concentration (Pareto) and year-over-year peak shift."""
        dpos  = daily[daily["revenue"] > 0]["revenue"].sort_values(ascending=False)
        top20 = max(int(len(dpos) * 0.20), 1)
        conc  = (
            round((dpos.iloc[:top20].sum() / dpos.sum()) * 100, 1)
            if not dpos.empty else 0.0
        )

        yoy: Dict[int, str] = {}
        for yr in sorted(ts.index.year.unique()):
            sub = ts[ts.index.year == yr]
            if len(sub) >= 28:
                pm = sub.groupby(sub.index.month)["revenue"].mean().idxmax()
                yoy[int(yr)] = MONTHS[int(pm) - 1]
        shift = len(set(yoy.values())) > 1 if len(yoy) >= 2 else False

        return {
            "revenue_concentration_pct":  conc,
            "top_20pct_days_count":        top20,
            "total_active_days":           int(len(dpos)),
            "seasonality_shift_detected":  shift,
            "yearly_peak_months":          yoy,
        }

    # ── metadata ───────────────────────────────────────────────────────────

    @staticmethod
    def _metadata(
        df:      pd.DataFrame,
        ts:      pd.DataFrame,
        monthly: pd.DataFrame,
    ) -> Dict[str, Any]:
        """Descriptive metadata about the dataset and analysis window."""
        return {
            "generated_at":    datetime.now().isoformat(timespec="seconds"),
            "data_start":      str(ts.index.min().date()),
            "data_end":        str(ts.index.max().date()),
            "total_days":      int((ts.index.max() - ts.index.min()).days),
            "calendar_months": int(len(monthly)),
            "total_revenue":   round(float(ts["revenue"].sum()), 2),
            "avg_daily_rev":   round(float(ts["revenue"].mean()), 2),
            "avg_monthly_rev": round(float(monthly["revenue"].mean()), 2),
            "total_orders":    int(len(df)),
        }


## 5. Insight Template Blueprints
Three f-string templates for different audiences:
- **Template A** — Executive Brief (store owner)
- **Template B** — Analytical Deep-Dive (analyst)
- **Template C** — Urgent Action Report (triggered on decline/anomalies)


In [ ]:
TEMPLATE_A = """\
╔══════════════════════════════════════════════════════════╗
║         DataSpeak — Executive Briefing                  ║
║  Period : {data_start}  →  {data_end}                   ║
╚══════════════════════════════════════════════════════════╝

📊 TREND       {trend_emoji}  Revenue is {direction}.
               Total change  : {pct_change:+.1f}%   |  Strength: {trend_strength}
               Monthly range : ${first_month:,.0f}  →  ${last_month:,.0f}

🗓  SEASONALITY Peak month    : {peak_month}  (+{peak_month_pct:.0f}% above average)
               Weakest month : {low_month}  (-{low_month_pct:.0f}% below average)
               Best day      : {peak_day}   |  Worst day: {low_day}
               Season data   : {season_reliability_note}

⚡  ANOMALIES   {anomaly_count} significant event(s) detected
               Revenue impact : ${anomaly_impact:,.0f}
               Latest event   : {anomaly_headline}

🔮 FORECAST    Next 7 days   →  ${next_week:,.0f}
               Next 30 days  →  ${next_month:,.0f}
               Confidence    :  {reliability}  (R²={r2:.3f})

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡  TOP PRIORITY: {top_priority}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

TEMPLATE_B = """\
╔══════════════════════════════════════════════════════════╗
║     DataSpeak — Analytical Deep-Dive Report             ║
║  Period : {data_start}  →  {data_end}                   ║
║  Total Revenue : ${total_revenue:,.0f}                   ║
╚══════════════════════════════════════════════════════════╝

━━ 1. TREND ANALYSIS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Direction    : {direction}  ({pct_change:+.1f}% over full period)
{baseline_note}
Slope        : {slope:+.2f} /month   (${slope_day:+.2f} /day)
R²           : {r2:.4f}   →  {r2_interpretation}
P-value      : {p_value:.4f}   →  {significance_note}
Stationarity : {stationarity_note}
Data months  : {data_months}

Interpretation:
{trend_interpretation}

━━ 2. SEASONALITY PROFILE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reliability  : {season_reliability_note}

Monthly Seasonality Indices  (1.00 = observed average)
{monthly_table}

Weekday Seasonality Indices
{weekday_table}

Key findings
  • {peak_month} is the dominant month    (index {peak_m_idx:.3f}  →  +{peak_month_pct:.0f}% above baseline)
  • {low_month} is the weakest month      (index {low_m_idx:.3f}  →  -{low_month_pct:.0f}% below baseline)
  • {peak_day} is the strongest weekday   (index {peak_d_idx:.3f})
  • {low_day} is the weakest weekday      (index {low_d_idx:.3f})
  • Seasonality shift across years        : {shift_note}

━━ 3. ANOMALY DETECTION ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Method    : Z-score on level-regression residuals (residual CV threshold)
Threshold : {anomaly_threshold:.2f}σ  (adaptive — based on residual CV)
Detected  : {anomaly_count}  ({spike_count} spikes  |  {drop_count} drops)
Impact    : ${anomaly_impact:,.0f}

{anomaly_detail_block}

━━ 4. FORECAST  (next 4 weeks) ━━━━━━━━━━━━━━━━━━━━━━━━━━

Method       : {forecast_method}
Reliability  : {reliability}  |  R²-basis = {r2:.3f}
Confidence   : 80%  |  Season correction applied: {season_applied}

{forecast_table}

Note: {forecast_caveat}

━━ 5. ADVANCED CHECKS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Revenue concentration  :  top 20% of days  →  {concentration:.1f}% of revenue
Pareto assessment      :  {pareto_note}
"""

TEMPLATE_C = """\
╔══════════════════════════════════════════════════════════╗
║  🚨  DataSpeak — Urgent Action Report                   ║
║  Triggered : {trigger_reason}                           ║
║  Period    : {data_start}  →  {data_end}                ║
╚══════════════════════════════════════════════════════════╝

SITUATION SUMMARY
{situation_summary}

━━ PRIORITY 1 — REVENUE TREND ━━━━━━━━━━━━━━━━━━━━━━━━━━━

{trend_risk_block}

━━ PRIORITY 2 — ANOMALY EVENTS ━━━━━━━━━━━━━━━━━━━━━━━━━━

{anomaly_checklist}

━━ PRIORITY 3 — DEFENSIVE ANCHORS ━━━━━━━━━━━━━━━━━━━━━━━

Strongest weekday  :  {peak_day}s  (+{peak_day_pct:.0f}% above weekly avg)
  → Concentrate promotions, email sends, and ad budget on {peak_day}s.

Peak month ahead   :  {peak_month}  (index {peak_m_idx:.2f}×  →  +{peak_month_pct:.0f}% above baseline)
  → Begin inventory and campaign preparation now.

━━ FORECAST  (worst / base / best) ━━━━━━━━━━━━━━━━━━━━━━

{forecast_table}
Total 30-day:  ${next_month:,.0f}  (reliability: {reliability})

━━ 30-DAY ACTION PLAN ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{action_plan}
"""


## 6. TemplateRenderer — Business Language Layer
Translates raw numerical facts → human-readable business language.  
All heuristic text logic lives here — `TimeSeriesEngine` stays purely numerical.


In [ ]:
class TemplateRenderer:
    """
    Translate raw engine facts into the three audience templates.

    All heuristic business-language logic lives here so that
    ``TimeSeriesEngine`` stays purely numerical.
    """

    def __init__(self, thresholds: Optional[Dict[str, Any]] = None) -> None:
        self.t = _get_thresholds(thresholds)

    def render_all(self, E: Dict[str, Any]) -> Dict[str, str]:
        """
        Build and return all three rendered templates plus the alert flag.

        Returns
        -------
        {
            "template_A": str,
            "template_B": str,
            "template_C": str,
            "should_alert": bool,   # BUG-FIX: surfaced here, not via internal call
        }
        """
        ctx         = self._build_context(E)
        should_alert = self._should_alert(E)
        return {
            "template_A":    TEMPLATE_A.format(**ctx),
            "template_B":    TEMPLATE_B.format(**ctx),
            "template_C":    TEMPLATE_C.format(**ctx),
            "should_alert":  should_alert,
        }

    # ── context builder ────────────────────────────────────────────────────

    def _build_context(self, E: Dict[str, Any]) -> Dict[str, Any]:
        """Map engine output keys → template placeholder values."""
        M   = E["metadata"]
        T   = E["trend"]
        S   = E["seasonality"]
        A   = E["anomalies"]
        F   = E["forecast"]
        ADV = E["advanced"]

        direction = T["direction"]
        r2        = T["r_squared"]
        pct       = T["pct_change_total"]
        sig       = T["is_significant"]

        # BUG-FIX: guard equal indices so tables don't double-label
        peak_m_idx = S["peak_month_index"]
        low_m_idx  = S["low_month_index"]
        peak_m_pct = (peak_m_idx - 1) * 100
        low_m_pct  = (1 - low_m_idx) * 100

        peak_d_idx = S["peak_weekday_index"]
        peak_d_pct = (peak_d_idx - 1) * 100

        anomaly_headline = (
            f"{A['anomaly_details'][0]['type'].upper()} on "
            f"{A['anomaly_details'][0]['period']}  "
            f"(${A['anomaly_details'][0]['actual_revenue']:,.0f} actual vs "
            f"${A['anomaly_details'][0]['expected_revenue']:,.0f} expected)"
            if A["anomaly_details"]
            else "None — revenue within expected bounds"
        )

        # BUG-FIX: pct_change baseline note
        bm = T["pct_change_baseline_month"]
        baseline_note = (
            f"  ⚠  % change baseline adjusted to month {bm + 1} (first non-zero month)."
            if bm > 0
            else ""
        )

        return {
            # metadata
            "data_start":           M["data_start"],
            "data_end":             M["data_end"],
            "total_revenue":        M["total_revenue"],
            "data_months":          T["data_months"],
            # trend
            "trend_emoji":          self._trend_emoji(direction, r2, sig),
            "direction":            direction,
            "pct_change":           pct,
            "baseline_note":        baseline_note,
            "slope":                T["slope_per_month"],
            "slope_day":            T["slope_per_day"],
            "r2":                   r2,
            "p_value":              T["p_value"],
            "first_month":          T["first_month_revenue"],
            "last_month":           T["last_month_revenue"],
            "trend_strength":       self._strength_label(r2, sig),
            "r2_interpretation":    self._r2_label(r2),
            "significance_note":    self._sig_note(T),
            "stationarity_note":    self._stat_note(T),
            "trend_interpretation": self._trend_narrative(T, M),
            "trend_risk_block":     self._trend_risk(T, direction, r2),
            # seasonality
            "peak_month":           S["peak_month"],
            "peak_month_pct":       peak_m_pct,
            "peak_m_idx":           peak_m_idx,
            "low_month":            S["low_month"],
            "low_month_pct":        low_m_pct,
            "low_m_idx":            low_m_idx,
            "peak_day":             S["peak_weekday"],
            "peak_d_idx":           peak_d_idx,
            "peak_day_pct":         peak_d_pct,
            "low_day":              S["low_weekday"],
            "low_d_idx":            S["low_weekday_index"],
            "monthly_table":        self._monthly_table(S["monthly_indices"]),
            "weekday_table":        self._weekday_table(S["weekday_indices"]),
            "shift_note":           self._shift_note(ADV),
            "season_reliability_note": self._season_reliability(S),
            # anomalies
            "anomaly_count":        A["total_detected"],
            "anomaly_impact":       A["revenue_impact"],
            "anomaly_headline":     anomaly_headline,
            "anomaly_threshold":    A["threshold_used"],
            "spike_count":          A["spike_count"],
            "drop_count":           A["drop_count"],
            "anomaly_detail_block": self._anomaly_block(A["anomaly_details"]),
            "anomaly_checklist":    self._anomaly_checklist(A["anomaly_details"]),
            # forecast
            "next_week":            F["next_week_estimate"],
            "next_month":           F["next_month_estimate"],
            "reliability":          F["reliability"],
            "forecast_method":      F["method"],
            "season_applied":       "Yes" if F["seasonality_applied"] else "No (insufficient data)",
            "forecast_table":       self._forecast_table(F["predictions"]),
            "forecast_caveat":      self._forecast_caveat(r2, F["reliability"]),
            # advanced
            "concentration":        ADV["revenue_concentration_pct"],
            "pareto_note":          self._pareto_note(ADV["revenue_concentration_pct"]),
            # composite
            "top_priority":      self._top_priority(T, A, S, direction),
            "trigger_reason":    self._trigger_reason(T, A, direction),
            "situation_summary": self._situation_summary(M, T, A, F, direction, pct),
            "action_plan":       self._action_plan(T, S, A, ADV, direction),
        }

    # ── sub-builders ──────────────────────────────────────────────────────

    @staticmethod
    def _trend_emoji(direction: str, r2: float, sig: bool) -> str:
        if not sig:
            return "➡️ "
        if direction == "increasing":
            return "📈" if r2 >= 0.45 else "↗️ "
        if direction == "decreasing":
            return "📉" if r2 >= 0.45 else "↘️ "
        return "➡️ "

    @staticmethod
    def _strength_label(r2: float, sig: bool) -> str:
        if not sig:
            return "not significant (noise)"
        if r2 >= 0.75:
            return "strong"
        if r2 >= 0.45:
            return "moderate"
        if r2 >= 0.20:
            return "weak"
        return "very weak"

    @staticmethod
    def _r2_label(r2: float) -> str:
        if r2 >= 0.75:
            return "Strong — trend explains most revenue variance"
        if r2 >= 0.45:
            return "Moderate — trend visible but noisy"
        if r2 >= 0.20:
            return "Weak — high volatility around the trend line"
        return "Very weak — revenue changes are largely random"

    @staticmethod
    def _sig_note(T: Dict[str, Any]) -> str:
        if T["is_significant"]:
            return "✅ Significant (p < 0.05) — direction is real, not noise"
        return (
            f"❌ Not significant (p = {T['p_value']:.3f} > 0.05) — "
            "treat direction as noise"
        )

    @staticmethod
    def _stat_note(T: Dict[str, Any]) -> str:
        if T["direction"] == "insufficient_data":
            return f"⚠  {T.get('_warning', 'Insufficient data')}"
        if T["is_stationary"]:
            return "Series is STATIONARY — regression on raw levels is valid"
        return "Series is NON-STATIONARY — regression run on first differences"

    @staticmethod
    def _trend_narrative(T: Dict[str, Any], M: Dict[str, Any]) -> str:
        if T["direction"] == "insufficient_data":
            return f"  ⚠  {T.get('_warning', 'Insufficient data for trend analysis.')}"
        d, pct, slope, r2 = (
            T["direction"], T["pct_change_total"],
            T["slope_per_month"], T["r_squared"],
        )
        sig_txt      = "statistically significant" if T["is_significant"] else "NOT statistically significant"
        variance_txt = "seasonal cycles and anomalies dominate" if r2 < 0.45 else "the trend is consistent"
        return (
            f"  Revenue has shown a {d} trajectory of {pct:+.1f}% "
            f"over {M['total_days']} days. The slope of "
            f"${slope:+.2f}/month is {sig_txt} "
            f"(p = {T['p_value']:.4f}). With R² = {r2:.3f}, {variance_txt}."
        )

    @staticmethod
    def _season_reliability(S: Dict[str, Any]) -> str:
        """BUG-FIX: surface the new season reliability flag."""
        if S["reliable"]:
            return f"✅ Reliable ({S['distinct_months']} distinct months observed)"
        return (
            f"⚠  Informative only — only {S['distinct_months']} distinct month(s) "
            "observed. Indices should not be used to scale a forecast."
        )

    @staticmethod
    def _monthly_table(m_idx: Dict[str, float]) -> str:
        if not m_idx:
            return "  No monthly data available."
        peak_v = max(m_idx.values())
        low_v  = min(m_idx.values())
        equal  = (peak_v == low_v)       # BUG-FIX: equal-revenue edge case
        lines  = []
        for month, idx in m_idx.items():
            bar = "█" * max(int(idx * 10), 0)
            if equal:
                tag = ""
            elif idx == peak_v:
                tag = " ← PEAK"
            elif idx == low_v:
                tag = " ← LOW"
            else:
                tag = ""
            lines.append(f"  {month:>3}  {idx:.3f}  {bar}{tag}")
        return "\n".join(lines)

    @staticmethod
    def _weekday_table(w_idx: Dict[str, float]) -> str:
        if not w_idx:
            return "  No weekday data available."
        peak_v = max(w_idx.values())
        low_v  = min(w_idx.values())
        equal  = (peak_v == low_v)       # BUG-FIX: equal-revenue edge case
        lines  = []
        for day, idx in w_idx.items():
            bar = "█" * max(int(idx * 10), 0)
            if equal:
                tag = ""
            elif idx == peak_v:
                tag = " ← PEAK"
            elif idx == low_v:
                tag = " ← LOW"
            else:
                tag = ""
            lines.append(f"  {day:>3}  {idx:.3f}  {bar}{tag}")
        return "\n".join(lines)

    @staticmethod
    def _shift_note(ADV: Dict[str, Any]) -> str:
        if ADV["seasonality_shift_detected"]:
            history = "  →  ".join(f"{yr}: {mo}" for yr, mo in ADV["yearly_peak_months"].items())
            return f"⚠  SHIFT DETECTED  ({history})"
        if ADV["yearly_peak_months"]:
            months = list(ADV["yearly_peak_months"].values())
            return f"✅  Stable — peak month is consistently {months[0]}"
        return "Insufficient multi-year data"

    @staticmethod
    def _anomaly_block(details: List[Dict[str, Any]]) -> str:
        if not details:
            return "  ✅  No anomalies — revenue within expected bounds throughout."
        lines = []
        for a in details:
            icon = "🔺" if a["type"] == "spike" else "🔻"
            hyp  = (
                "Promo event, bulk order, or data error — verify source."
                if a["type"] == "spike"
                else "Checkout failure, stockout, or ad blackout — audit ops logs."
            )
            lines.append(
                f"\n  {icon} [{a['severity'].upper()}] "
                f"{a['type'].upper()} — {a['period']}\n"
                f"     Actual    : ${a['actual_revenue']:>12,.0f}\n"
                f"     Expected  : ${a['expected_revenue']:>12,.0f}\n"
                f"     Z-score   : {a['zscore']:+.3f}  |  "
                f"Deviation: {a['pct_deviation']:+.0f}%\n"
                f"     Hypothesis: {hyp}"
            )
        return "\n".join(lines)

    @staticmethod
    def _anomaly_checklist(details: List[Dict[str, Any]]) -> str:
        if not details:
            return "  ✅  No anomaly events require investigation."
        lines = []
        for i, a in enumerate(details, start=1):
            icon = "🔺 SPIKE" if a["type"] == "spike" else "🔻 DROP"
            lines.append(
                f"\n  Event {i}: {icon} on {a['period']}\n"
                f"  Actual ${a['actual_revenue']:,.0f} vs "
                f"expected ${a['expected_revenue']:,.0f} "
                f"({a['pct_deviation']:+.0f}%,  z = {a['zscore']:+.2f})"
            )
            steps = (
                [
                    "Check marketing calendar — was a promo running?",
                    "Verify no duplicate transactions in raw order log",
                    "Confirm bulk/wholesale orders were intentional",
                    "If legitimate: document driver and plan to replicate",
                ]
                if a["type"] == "spike"
                else [
                    "Pull payment gateway error logs for this period",
                    "Check website uptime / checkout error rate",
                    "Verify ad spend was active (Meta / Google Ads)",
                    "Check inventory levels for top-selling SKUs",
                    "Review customer support tickets from this period",
                ]
            )
            for step in steps:
                lines.append(f"    □  {step}")
        return "\n".join(lines)

    @staticmethod
    def _forecast_table(predictions: List[Dict[str, Any]]) -> str:
        rows = [
            "  Date           Forecast     Lower (80%)   Upper (80%)   Method",
            "  " + "─" * 72,
        ]
        for p in predictions:
            rows.append(
                f"  {p['date']}   "
                f"${p['predicted_revenue']:>10,.0f}   "
                f"${p['lower_bound']:>10,.0f}   "
                f"${p['upper_bound']:>10,.0f}   "
                f"{p['method_used']}"
            )
        return "\n".join(rows)

    @staticmethod
    def _forecast_caveat(r2: float, reliability: str) -> str:
        if reliability == "high":
            return f"R² = {r2:.3f} — model captures most variance; forecast is reliable."
        if reliability == "medium":
            return f"R² = {r2:.3f} — moderate fit. Actual results may vary ±20%."
        return (
            f"R² = {r2:.3f} — low fit. "
            "Forecast uses mean × seasonality index. Treat as directional guidance only."
        )

    @staticmethod
    def _pareto_note(conc: float) -> str:
        if conc >= 80:
            return "🔴 HIGH fragility — a bad peak day has outsized monthly impact."
        if conc >= 60:
            return "🟡 Moderate concentration — normal for e-commerce."
        return "🟢 Healthy distribution — revenue spread across many days."

    @staticmethod
    def _top_priority(
        T: Dict[str, Any],
        A: Dict[str, Any],
        S: Dict[str, Any],
        direction: str,
    ) -> str:
        if direction == "decreasing" and T["is_significant"] and A["total_detected"] > 0:
            return (
                f"Revenue is declining AND {A['total_detected']} anomaly event(s) "
                "require investigation. Start with anomaly audit."
            )
        if direction == "decreasing" and T["is_significant"]:
            return (
                f"Confirmed revenue decline. Activate win-back campaign "
                f"before the {S['peak_month']} seasonal window."
            )
        if A["total_detected"] > 0:
            return (
                f"{A['total_detected']} revenue anomaly event(s) detected "
                f"(${A['revenue_impact']:,.0f} impact). Investigate immediately."
            )
        if direction == "increasing" and T["is_significant"]:
            return (
                f"Revenue is growing. Scale inventory ahead of {S['peak_month']} "
                "and protect the acquisition funnel."
            )
        return (
            f"Revenue is stable. Invest in conversion optimisation "
            f"on {S['low_weekday']}s to lift the weekly floor."
        )

    @staticmethod
    def _trigger_reason(
        T: Dict[str, Any],
        A: Dict[str, Any],
        direction: str,
    ) -> str:
        triggers = []
        if direction == "decreasing" and T["is_significant"]:
            triggers.append("Confirmed declining revenue trend")
        if A["total_detected"] > 0:
            triggers.append(
                f"{A['total_detected']} anomaly event(s) — "
                f"${A['revenue_impact']:,.0f} impact"
            )
        return " | ".join(triggers) if triggers else "Scheduled automated review"

    @staticmethod
    def _situation_summary(
        M: Dict[str, Any],
        T: Dict[str, Any],
        A: Dict[str, Any],
        F: Dict[str, Any],
        direction: str,
        pct: float,
    ) -> str:
        return (
            f"  Your store generated ${M['total_revenue']:,.0f} over "
            f"{M['total_days']} days "
            f"({M['data_start']} to {M['data_end']}). "
            f"Revenue is currently {direction} ({pct:+.1f}% over the full period). "
            f"{A['total_detected']} anomaly(ies) detected — "
            f"${A['revenue_impact']:,.0f} total revenue deviation. "
            f"30-day forecast: ${F['next_month_estimate']:,.0f} "
            f"(reliability: {F['reliability']})."
        )

    @staticmethod
    def _trend_risk(
        T: Dict[str, Any],
        direction: str,
        r2: float,
    ) -> str:
        if T["direction"] == "insufficient_data":
            return f"  ⚠  {T.get('_warning', 'Insufficient data.')}"
        if not T["is_significant"]:
            return (
                f"  Trend is NOT statistically significant "
                f"(p = {T['p_value']:.3f}). "
                "Revenue movements are likely driven by seasonality and anomalies — "
                "monitor 2 more periods before escalating."
            )
        if direction == "decreasing":
            return (
                f"  ⚠  Confirmed decline  "
                f"(p = {T['p_value']:.4f}, R² = {r2:.3f}). "
                f"Revenue is falling at "
                f"${abs(T['slope_per_month']):,.2f}/month. "
                "Without intervention this trajectory continues into the forecast window."
            )
        return (
            f"  ✅  Trend is positive and significant  "
            f"(p = {T['p_value']:.4f}). "
            f"Revenue growing at ${T['slope_per_month']:,.2f}/month."
        )

    @staticmethod
    def _action_plan(
        T:         Dict[str, Any],
        S:         Dict[str, Any],
        A:         Dict[str, Any],
        ADV:       Dict[str, Any],
        direction: str,
    ) -> str:
        steps = []
        n     = 1
        if A["total_detected"] > 0:
            steps.append(
                f"  {n}. [IMMEDIATE] Audit the {A['total_detected']} anomaly period(s) "
                "listed above. Pull raw order logs and cross-reference the marketing calendar."
            )
            n += 1
        if A.get("spike_count", 0) > 0:
            steps.append(
                f"  {n}. [THIS WEEK] Identify the driver behind the revenue spike(s) "
                f"and schedule a repeat before {S['peak_month']}."
            )
            n += 1
        if direction == "decreasing" and T["is_significant"]:
            steps.append(
                f"  {n}. [THIS WEEK] Launch a win-back campaign for "
                "lapsed customers. A 15% re-engagement discount costs "
                "less than acquiring new customers."
            )
            n += 1
        steps.append(
            f"  {n}. [THIS MONTH] Increase marketing budget for "
            f"{S['peak_month']} — "
            f"index {S['peak_month_index']:.2f}× means every $1 returns "
            f"{S['peak_month_index']:.1f}× the average monthly ROI."
        )
        n += 1
        fri_idx = S["weekday_indices"].get("Fri", 1.0)
        sat_idx = S["weekday_indices"].get("Sat", 1.0)
        steps.append(
            f"  {n}. [ONGOING] Prioritise Fri/Sat capacity — "
            f"indices {fri_idx:.2f}× / {sat_idx:.2f}×."
        )
        n += 1
        if ADV["seasonality_shift_detected"]:
            steps.append(
                f"  {n}. [STRATEGIC] Seasonality pattern has shifted across years — "
                "update campaign calendar; historical peak assumptions may no longer apply."
            )
        return "\n".join(steps)

    @staticmethod
    def _should_alert(E: Dict[str, Any]) -> bool:
        """True when Template C should be surfaced to the user."""
        return (
            E["anomalies"]["total_detected"] > 0
            or E["trend"]["direction"] == "decreasing"
        )


## 7. Validation
Hard validation before engine runs.  
Raises `ValueError` on < 3 calendar months or < 60-day span.  
Emits warnings (not errors) for < 6 months.


In [ ]:
def validate_dataframe(df: pd.DataFrame, filename: str = "") -> None:
    """
    Raise ``ValueError`` with a clear message if *df* cannot be safely
    processed by the engine.

    Validates calendar months (not raw row count) and minimum date span.
    Warnings are emitted (not errors) when data is below the recommended
    threshold but above the hard minimum.
    """
    prefix = f"[{filename}] " if filename else ""
    errors: List[str] = []

    for col in ("order_date", "revenue"):
        if col not in df.columns:
            errors.append(f"Missing required column: '{col}'")

    if "order_date" in df.columns:
        try:
            dates    = pd.to_datetime(df["order_date"], errors="coerce")
            valid    = dates.dropna()
            if valid.empty:
                errors.append("No parseable date values found in 'order_date'.")
            else:
                months   = int(valid.dt.to_period("M").nunique())
                day_span = int((valid.max() - valid.min()).days)
                min_m    = _DEFAULT_THRESHOLDS["validation"]["min_calendar_months"]
                min_days = _DEFAULT_THRESHOLDS["validation"]["min_days_span"]
                rec_m    = _DEFAULT_THRESHOLDS["validation"]["recommended_months"]

                if months < min_m:
                    errors.append(
                        f"Only {months} calendar month(s) detected. "
                        f"Hard minimum is {min_m} months."
                    )
                elif months < rec_m:
                    logger.warning(
                        "%s%d months of data — recommend %d+ for reliable trend detection.",
                        prefix, months, rec_m,
                    )
                    print(
                        f"  ⚠  {prefix}{months} months of data — "
                        f"recommend {rec_m}+ for reliable trend detection."
                    )

                if day_span < min_days:
                    errors.append(
                        f"Date range is only {day_span} days. "
                        f"Minimum is {min_days} days."
                    )
        except Exception as exc:
            errors.append(f"order_date cannot be parsed: {exc}")

    if errors:
        raise ValueError(
            f"{prefix}Validation failed:\n" +
            "\n".join(f"  ❌  {e}" for e in errors)
        )


## 8. TimeSeriesAgent — Orchestration + LLM Layer
Single entry point for the entire pipeline.  
Connects to Groq/Llama-3.3 and generates the executive LLM summary.  
```python
agent = TimeSeriesAgent()  # reads GROQ_API_KEY from env
report = agent.run(raw_df, filename='store_a.csv')
print(report['template_A'])   # Executive brief
print(report['llm_summary'])  # LLM commentary
```


In [ ]:
class TimeSeriesAgent:
    """
    Single entry point for the entire DataSpeak Time Series pipeline.

    Parameters
    ----------
    groq_api_key:
        Groq API key.  If *None*, falls back to the ``GROQ_API_KEY``
        environment variable.  Never hard-code this value.
    thresholds:
        Partial or full sensitivity configuration dict.
        Only the keys you supply are overridden; the rest stay at defaults.

    Usage
    -----
    ::

        agent  = TimeSeriesAgent()
        report = agent.run(raw_df, filename="store_a.csv")

        print(report["template_A"])          # Executive brief
        print(report["template_B"])          # Deep-dive analytics
        print(report["llm_summary"])         # LLM executive commentary
        print(report["should_alert"])        # bool — show Template C?

        # For Motawea (Visualization Agent):
        daily_df   = report["engine_output"]["series"]["daily"]
        monthly_df = report["engine_output"]["series"]["monthly"]
    """

    _SYSTEM_PROMPT = (
        "You are a Senior Retail Business Analyst writing an executive briefing "
        "for a non-technical e-commerce store owner.\n\n"
        "You will receive a JSON object with precise numerical facts about the "
        "store's performance over a specific period.  Your task:\n\n"
        "  1. Write ONE cohesive professional paragraph of 150–180 words in second "
        "person ('Your revenue…', 'Your strongest month…').\n"
        "  2. Reference only the numbers in the JSON — never invent data.\n"
        "  3. Mention trend direction, the top anomaly (if any), the peak seasonal "
        "period, and the 30-day forecast.\n"
        "  4. After the paragraph, output exactly 3 numbered, highly actionable "
        "strategic business recommendations in bullet form.\n"
        "     Each recommendation must be concrete (what to do, when, and why).\n"
        "  5. Tone: senior consultant briefing a busy entrepreneur — clear, direct, "
        "no jargon.\n"
        "  6. Output plain text only — no markdown, no asterisks, no headers.\n"
    )

    def __init__(
        self,
        groq_api_key: Optional[str] = None,
        thresholds:   Optional[Dict[str, Any]] = None,
    ) -> None:
        t = _get_thresholds(thresholds)
        self.engine   = TimeSeriesEngine(t)
        self.renderer = TemplateRenderer(t)

        key       = groq_api_key or os.getenv("GROQ_API_KEY")
        self.llm: Optional[ChatGroq] = None
        if key:
            self.llm = ChatGroq(
                model="llama-3.3-70b-versatile",
                groq_api_key=key,
                temperature=0.3,
            )
            logger.info("LLM initialised (Groq / llama-3.3-70b-versatile).")
        else:
            logger.warning("GROQ_API_KEY not found — LLM summaries will be skipped.")
            print("  ⚠  GROQ_API_KEY not found. LLM summary will be skipped.")

    # ── public pipeline ────────────────────────────────────────────────────

    def run(
        self,
        raw_df:               pd.DataFrame,
        filename:             str  = "",
        generate_llm_summary: bool = True,
    ) -> Dict[str, Any]:
        """
        Execute the full pipeline on *raw_df* and return a results dict.

        Parameters
        ----------
        raw_df:
            Raw DataFrame loaded from CSV (any schema).
        filename:
            Optional label for log messages.
        generate_llm_summary:
            Set *False* to skip the Groq API call (faster for batch testing).

        Returns
        -------
        {
            "template_A":    str,
            "template_B":    str,
            "template_C":    str,
            "should_alert":  bool,    ← BUG-FIX: now lives on the report dict
            "llm_summary":   str | None,
            "engine_output": dict,    ← full facts + series DataFrames
            "facts_json":    dict,    ← engine output without DataFrames (JSON-safe)
        }
        """
        tag = f"[{filename}] " if filename else ""
        logger.info("%sPipeline started.", tag)

        # 1. Schema normalisation
        print(f"{tag}🔧  Standardising schema…")
        clean_df, returns_df = SchemaAdapter.standardize(raw_df, filename)

        # 2. Validation
        print(f"{tag}✔   Validating dataset…")
        validate_dataframe(clean_df, filename)

        # 3. Engine
        print(f"{tag}⚙   Running Time Series Engine…")
        engine_out = self.engine.run(clean_df)

        if returns_df is not None and not returns_df.empty:
            engine_out["returns_summary"] = {
                "count":         int(len(returns_df)),
                "total_returns": round(float(returns_df["revenue"].sum()), 2),
            }

        # 4. Template rendering
        print(f"{tag}📝  Rendering templates…")
        rendered = self.renderer.render_all(engine_out)

        # 5. LLM reasoning layer
        llm_summary: Optional[str] = None
        if generate_llm_summary and self.llm is not None:
            print(f"{tag}🤖  Generating LLM executive summary (Groq / Llama-3.3)…")
            llm_summary = self._llm_summary(engine_out)
        else:
            print(f"{tag}ℹ   LLM summary skipped.")

        # 6. Assemble JSON-safe facts (strip DataFrames)
        facts_json = {k: v for k, v in engine_out.items() if k != "series"}

        logger.info("%sPipeline complete.", tag)
        print(f"{tag}✅  Pipeline complete.\n")

        return {
            "template_A":    rendered["template_A"],
            "template_B":    rendered["template_B"],
            "template_C":    rendered["template_C"],
            "should_alert":  rendered["should_alert"],    # BUG-FIX
            "llm_summary":   llm_summary,
            "engine_output": engine_out,
            "facts_json":    facts_json,
        }

    # ── LLM reasoning layer ────────────────────────────────────────────────

    def _llm_summary(self, E: Dict[str, Any]) -> str:
        """
        Call Groq / Llama-3.3 with structured JSON context.

        Passing structured JSON (not the rendered box-drawing template) avoids
        the LLM needing to parse visual formatting and prevents hallucination
        of numbers not present in the facts.
        """
        T = E["trend"]
        S = E["seasonality"]
        A = E["anomalies"]
        F = E["forecast"]
        M = E["metadata"]

        facts = {
            "period":                    f"{M['data_start']} to {M['data_end']}",
            "total_revenue_usd":         M["total_revenue"],
            "avg_monthly_rev_usd":       M["avg_monthly_rev"],
            "trend_direction":           T["direction"],
            "trend_slope_monthly":       T["slope_per_month"],
            "trend_r_squared":           T["r_squared"],
            "pct_change_total":          T["pct_change_total"],
            "pct_change_baseline_month": T["pct_change_baseline_month"],
            "statistically_significant": T["is_significant"],
            "peak_month":                S["peak_month"],
            "peak_month_index":          S["peak_month_index"],
            "low_month":                 S["low_month"],
            "peak_weekday":              S["peak_weekday"],
            "seasonality_reliable":      S["reliable"],
            "anomalies_detected":        A["total_detected"],
            "anomaly_revenue_impact_usd":A["revenue_impact"],
            "anomaly_periods":           A["anomaly_periods"],
            "forecast_next_7d":          F["next_week_estimate"],
            "forecast_next_30d":         F["next_month_estimate"],
            "forecast_reliability":      F["reliability"],
            "seasonality_applied_to_forecast": F["seasonality_applied"],
        }

        user_msg = (
            "Here are the precise numerical facts for this store:\n\n"
            + json.dumps(facts, indent=2)
            + "\n\nWrite the executive briefing and 3 strategic recommendations now."
        )

        try:
            response = self.llm.invoke([
                SystemMessage(content=self._SYSTEM_PROMPT),
                HumanMessage(content=user_msg),
            ])
            return response.content.strip()
        except Exception as exc:
            logger.error("LLM call failed: %s", exc)
            return f"[LLM unavailable: {exc}]"


## 9. Helper Utilities
Internal helpers: `_suppress_statsmodels_warnings()`, `_print_divider()`, `_print_numeric_facts()`.


In [ ]:
@contextlib.contextmanager
def _suppress_statsmodels_warnings():
    """Scope statsmodels / numpy RuntimeWarnings to the ADF call only."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        warnings.filterwarnings("ignore", message=".*InterpolationWarning.*")
        yield


def _print_divider(label: str) -> None:
    width = 68
    print("\n" + "═" * width)
    print(f"  {label}")
    print("═" * width + "\n")


def _print_numeric_facts(facts_json: Dict[str, Any], tag: str = "") -> None:
    """
    Pretty-print the key numeric facts from the engine output.

    ── For Motawea ──────────────────────────────────────────────────────────
    The same dict is available at report["facts_json"] and can be passed
    directly to your charting functions.
    """
    M = facts_json.get("metadata", {})
    T = facts_json.get("trend",    {})
    F = facts_json.get("forecast", {})
    A = facts_json.get("anomalies",{})

    _print_divider(f"NUMERIC FACTS — {tag}")
    print(f"  Period          : {M.get('data_start')}  →  {M.get('data_end')}")
    print(f"  Total revenue   : ${M.get('total_revenue', 0):,.2f}")
    print(f"  Avg daily rev   : ${M.get('avg_daily_rev', 0):,.2f}")
    print(f"  Avg monthly rev : ${M.get('avg_monthly_rev', 0):,.2f}")
    print(f"  Calendar months : {M.get('calendar_months')}")
    print(f"  Total orders    : {M.get('total_orders', 0):,}")
    print()
    print(f"  Trend direction : {T.get('direction')}")
    print(f"  Slope /month    : ${T.get('slope_per_month', 0):+,.2f}")
    print(f"  R²              : {T.get('r_squared', 0):.4f}")
    print(f"  P-value         : {T.get('p_value', 1):.6f}")
    print(f"  Pct change      : {T.get('pct_change_total', 0):+.1f}%")
    print()
    print(f"  Anomalies       : {A.get('total_detected', 0)}")
    print(f"  Anomaly impact  : ${A.get('revenue_impact', 0):,.2f}")
    print()
    print(f"  Forecast 7d     : ${F.get('next_week_estimate', 0):,.2f}")
    print(f"  Forecast 30d    : ${F.get('next_month_estimate', 0):,.2f}")
    print(f"  Reliability     : {F.get('reliability')}")
    print()


## 10. Execution Pipeline
Top-level `process_csv()` and `main()` functions.  
`main()` loads `cleaned_data_1.csv` and `cleaned_data_2.csv` sequentially.


In [ ]:
def process_csv(
    filepath: str,
    agent:    TimeSeriesAgent,
    show_template_b: bool = True,
    show_template_c: bool = True,
) -> Optional[Dict[str, Any]]:
    """
    Load *filepath*, run the full agent pipeline, and print the report.

    Parameters
    ----------
    filepath:
        Absolute or relative path to the cleaned CSV file.
        (This is Farhat's output file — place it next to this script.)
    agent:
        Pre-initialised ``TimeSeriesAgent`` instance (one LLM connection).
    show_template_b:
        Print the Analytical Deep-Dive template.
    show_template_c:
        Print the Urgent Action template when triggered.

    Returns
    -------
    The full report dict (all templates, engine output, series DataFrames),
    or *None* if an error occurred.

    ── For Motawea (Visualization Agent) ────────────────────────────────────
    When this function returns a report dict you can immediately plot:

        daily_df   = report["engine_output"]["series"]["daily"]
        rolling_df = report["engine_output"]["series"]["rolling"]
        monthly_df = report["engine_output"]["series"]["monthly"]

    The anomaly periods in:
        report["engine_output"]["anomalies"]["anomaly_details"]
    each contain "period", "actual_revenue", "expected_revenue", "zscore" —
    all you need for chart annotations.

    The 4-week forecast is in:
        report["engine_output"]["forecast"]["predictions"]
    each entry has "date", "predicted_revenue", "lower_bound", "upper_bound".
    """
    filename = os.path.basename(filepath)
    _print_divider(f"PROCESSING: {filename}")

    # ── Load CSV ───────────────────────────────────────────────────────────
    if not os.path.exists(filepath):
        print(f"  ❌  File not found: {filepath}")
        logger.error("File not found: %s", filepath)
        return None

    try:
        raw_df = pd.read_csv(filepath)
        print(f"  Loaded {len(raw_df):,} rows  |  Columns: {list(raw_df.columns)}")
        logger.info("Loaded %s — %d rows", filename, len(raw_df))
    except Exception as exc:
        print(f"  ❌  Could not read CSV: {exc}")
        logger.error("Could not read CSV %s: %s", filepath, exc)
        return None

    # ── Run agent ──────────────────────────────────────────────────────────
    try:
        report = agent.run(
            raw_df=raw_df,
            filename=filename,
            generate_llm_summary=True,
        )
    except ValueError as exc:
        print(f"\n  ❌  Validation error:\n{exc}")
        logger.error("Validation error for %s: %s", filename, exc)
        return None
    except Exception as exc:
        print(f"\n  ❌  Pipeline error: {exc}")
        logger.exception("Unexpected pipeline error for %s", filename)
        raise   # re-raise so the full traceback is visible during development

    # ── Numeric facts ──────────────────────────────────────────────────────
    _print_numeric_facts(report["facts_json"], tag=filename)

    # ── Template A — always shown ─────────────────────────────────────────
    _print_divider(f"TEMPLATE A — Executive Brief  |  {filename}")
    print(report["template_A"])

    # ── Template B — deep dive ────────────────────────────────────────────
    if show_template_b:
        _print_divider(f"TEMPLATE B — Analytical Deep-Dive  |  {filename}")
        print(report["template_B"])

    # ── Template C — only when alert is triggered ─────────────────────────
    # BUG-FIX: should_alert is now read from the report dict, not via
    # an internal static method call that creates coupling.
    if show_template_c and report["should_alert"]:
        _print_divider(f"TEMPLATE C — Urgent Action Report  |  {filename}")
        print(report["template_C"])
    elif show_template_c:
        print(
            f"\n  ℹ   Template C not triggered for {filename} "
            "(no decline or anomalies detected)."
        )

    # ── LLM executive summary ─────────────────────────────────────────────
    if report["llm_summary"]:
        _print_divider(f"LLM EXECUTIVE SUMMARY  |  {filename}")
        print(report["llm_summary"])

    # ── DataFrame handoff note for Motawea ────────────────────────────────
    _print_divider(f"SERIES DATAFRAMES READY FOR VISUALIZATION  |  {filename}")
    series = report["engine_output"]["series"]
    for name, df in series.items():
        print(f"  report['engine_output']['series']['{name}']  — shape: {df.shape}")
    print()
    print("  Anomaly periods :", report["engine_output"]["anomalies"]["anomaly_periods"])
    fc_dates = [p["date"] for p in report["engine_output"]["forecast"]["predictions"]]
    print("  Forecast dates  :", fc_dates)

    return report


def main() -> None:
    """
    Sequential execution pipeline:

      1. Initialise one shared agent (single LLM connection).
      2. Process cleaned_data_1.csv  (Farhat's first output file)
      3. Process cleaned_data_2.csv  (Farhat's second output file)
      4. Print a final status summary.

    Environment
    -----------
    Set ``GROQ_API_KEY`` in your shell before running:
        export GROQ_API_KEY="gsk_..."

    The script will run without a Groq key — LLM summaries will be skipped
    and all template + numeric outputs will still be produced.
    """
    _print_divider("DataSpeak — Time Series & Forecasting Agent  |  Production v3.0")

    # ── Detect API key ────────────────────────────────────────────────────
    groq_key = os.getenv("GROQ_API_KEY")
    if not groq_key:
        print("  ⚠  GROQ_API_KEY not set — LLM summaries will be skipped.")
        print("     Set it with: export GROQ_API_KEY='gsk_...'")
    else:
        print("  ✅  GROQ_API_KEY detected.")

    # ── Initialise one shared agent ────────────────────────────────────────
    # مبدأ: نشيّل الـ agent مرة وحدة بس، وبنمرره لكل ملف —
    # ده بيوفر connection واحد للـ LLM بدل ما نعمل connection جديد لكل ملف.
    agent = TimeSeriesAgent(groq_api_key=groq_key)

    # ── Dataset paths ──────────────────────────────────────────────────────
    # Update these paths if Farhat places the cleaned files in a sub-folder.
    # Example: "output/cleaned_data_1.csv"
    datasets = [
        "cleaned_data_1.csv",
        "cleaned_data_2.csv",
    ]

    results: Dict[str, Optional[Dict[str, Any]]] = {}

    for path in datasets:
        # كل ملف بياخد نفس الـ agent — نفس الـ thresholds، نفس الـ LLM connection
        report = process_csv(
            filepath=path,
            agent=agent,
            show_template_b=True,
            show_template_c=True,
        )
        results[path] = report

    # ── Final status summary ───────────────────────────────────────────────
    _print_divider("PIPELINE COMPLETE — FINAL STATUS")
    for path, report in results.items():
        if report:
            alert = report.get("should_alert", False)
            alert_tag = "⚠  ALERT TRIGGERED" if alert else "✅  No Alert"
            print(f"  ✅  Success  —  {os.path.basename(path)}  |  {alert_tag}")
        else:
            print(f"  ❌  Failed   —  {os.path.basename(path)}")

    # ── Handoff note for Motawea ───────────────────────────────────────────
    # لموتوع: كل الـ DataFrames اللي محتاجها للـ visualization موجودة في:
    #   results["cleaned_data_1.csv"]["engine_output"]["series"]["daily"]
    #   results["cleaned_data_1.csv"]["engine_output"]["series"]["rolling"]
    #   results["cleaned_data_1.csv"]["engine_output"]["series"]["monthly"]
    #   results["cleaned_data_1.csv"]["engine_output"]["forecast"]["predictions"]
    #   results["cleaned_data_1.csv"]["engine_output"]["anomalies"]["anomaly_details"]
    print()
    print("  ── For Motawea (EDA / Visualization Agent) ──────────────────────────")
    print("  All series DataFrames are in: results[filepath]['engine_output']['series']")
    print("  Forecast predictions       : results[filepath]['engine_output']['forecast']")
    print("  Anomaly annotations        : results[filepath]['engine_output']['anomalies']")
    print()


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


---
# Demo: Run the Full Pipeline
## Option A — Use Your Real CSV Files
Place `cleaned_data_1.csv` and `cleaned_data_2.csv` next to this notebook, then run:


In [ ]:
import os

# ── Set your Groq API key (or set it in your shell: export GROQ_API_KEY='gsk_...')
# os.environ['GROQ_API_KEY'] = 'gsk_...'

groq_key = os.getenv('GROQ_API_KEY')
if not groq_key:
    print('⚠  GROQ_API_KEY not set — LLM summaries will be skipped.')
    print("  Set it with: os.environ['GROQ_API_KEY'] = 'gsk_...'")
else:
    print('✅  GROQ_API_KEY detected.')

# Initialise one shared agent (single LLM connection reused across all files)
agent = TimeSeriesAgent(groq_api_key=groq_key)


In [ ]:
# Process both CSV files from Farhat
results = {}
for path in ['cleaned_data_1.csv', 'cleaned_data_2.csv']:
    report = process_csv(filepath=path, agent=agent,
                         show_template_b=True, show_template_c=True)
    results[path] = report

# ── For Motawea: access the DataFrames ──────────────────────────────
# daily_df   = results['cleaned_data_1.csv']['engine_output']['series']['daily']
# rolling_df = results['cleaned_data_1.csv']['engine_output']['series']['rolling']
# monthly_df = results['cleaned_data_1.csv']['engine_output']['series']['monthly']
# forecast   = results['cleaned_data_1.csv']['engine_output']['forecast']['predictions']
# anomalies  = results['cleaned_data_1.csv']['engine_output']['anomalies']['anomaly_details']


## Option B — Synthetic Data Demo (No CSV Required)
Generates 2 years of realistic e-commerce data with injected anomalies.


In [ ]:
# ── Generate synthetic data ───────────────────────────────────────────
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)
dates     = pd.date_range('2023-01-01', '2024-12-31', freq='D')
n_days    = len(dates)
trend_c   = np.linspace(300, 700, n_days)
season_c  = 1 + 0.35 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)
noise     = np.random.normal(0, 55, n_days)
revenue   = (trend_c * season_c + noise).clip(0)

# Inject anomalies for detection testing
revenue[180:183] += 1500   # spike  — late June
revenue[450:452] -= 400    # drop   — mid-March year 2

synthetic_df = pd.DataFrame({'order_date': dates, 'revenue': revenue})
synthetic_df.to_csv('synthetic_store.csv', index=False)

print(f'✅  Generated {len(synthetic_df):,} daily records')
print(f'   Period: {dates[0].date()} → {dates[-1].date()}')
print(f'   Revenue range: ${revenue.min():.0f} – ${revenue.max():.0f}/day')
print(f'   Mean: ${revenue.mean():.0f}/day | Total: ${revenue.sum():,.0f}')
synthetic_df.head()


In [ ]:
import os

# Run the full pipeline on synthetic data
groq_key = os.getenv('GROQ_API_KEY')
agent    = TimeSeriesAgent(groq_api_key=groq_key)

report = agent.run(
    raw_df=synthetic_df,
    filename='synthetic_store.csv',
    generate_llm_summary=(groq_key is not None),
)


### Numeric Facts


In [ ]:
_print_numeric_facts(report['facts_json'], tag='Synthetic Store')


### Template A — Executive Brief


In [ ]:
print(report['template_A'])


### Template B — Analytical Deep-Dive


In [ ]:
print(report['template_B'])


### Template C — Urgent Action Report
Only printed when `should_alert == True` (declining revenue or anomalies detected).


In [ ]:
print(f"Alert triggered: {report['should_alert']}")
if report['should_alert']:
    print(report['template_C'])


### LLM Executive Summary


In [ ]:
if report['llm_summary']:
    print(report['llm_summary'])
else:
    print('LLM summary not generated (no GROQ_API_KEY).')


### Series DataFrames (for Motawea)
These are ready-to-plot DataFrames with a `DatetimeIndex`.


In [ ]:
series = report['engine_output']['series']
for name, df in series.items():
    print(f"  series['{name}']  shape: {df.shape}  cols: {list(df.columns)}")

print()
print('Anomaly periods:', report['engine_output']['anomalies']['anomaly_periods'])

fc = report['engine_output']['forecast']['predictions']
print('\nForecast predictions:')
for p in fc:
    print(f"  Week {p['week_number']}  {p['date']}  "
          f"${p['predicted_revenue']:,.0f}  "
          f"[${p['lower_bound']:,.0f} – ${p['upper_bound']:,.0f}]")


### Seasonality Indices


In [ ]:
import pandas as pd

s = report['engine_output']['seasonality']

print(f"Reliable: {s['reliable']}  ({s['distinct_months']} distinct months)")
print(f"Peak month: {s['peak_month']}  (index {s['peak_month_index']:.3f})")
print(f"Low  month: {s['low_month']}  (index {s['low_month_index']:.3f})")
print()

m_df = pd.DataFrame(list(s['monthly_indices'].items()), columns=['Month','Index'])
w_df = pd.DataFrame(list(s['weekday_indices'].items()), columns=['Day','Index'])

print('Monthly indices:')
display(m_df.set_index('Month').T)
print()
print('Weekday indices:')
display(w_df.set_index('Day').T)


### Anomaly Details


In [ ]:
anomalies = report['engine_output']['anomalies']
print(f"Detected: {anomalies['total_detected']}  "
      f"(spikes: {anomalies['spike_count']}  drops: {anomalies['drop_count']})")
print(f"Revenue impact: ${anomalies['revenue_impact']:,.2f}")
print(f"Threshold used: {anomalies['threshold_used']:.2f}σ")
print()

for a in anomalies['anomaly_details']:
    icon = '🔺' if a['type'] == 'spike' else '🔻'
    print(f"{icon} [{a['severity'].upper()}] {a['type'].upper()} — {a['period']}")
    print(f"   Actual   : ${a['actual_revenue']:>12,.0f}")
    print(f"   Expected : ${a['expected_revenue']:>12,.0f}")
    print(f"   Z-score  : {a['zscore']:+.3f}  |  Deviation: {a['pct_deviation']:+.0f}%")
    print()


---
# Forecast Evaluation & Visualization
Interactive Plotly chart with:
- Daily revenue bars
- 7-day and 30-day rolling averages with ±1σ bands
- OLS trend line (dashed)
- Anomaly markers (hover for details)
- 4-week forecast with proper OLS prediction interval
- Metric box: RMSE, MAE, MAPE, R², slope
- Monthly overview (anomaly months highlighted)


In [ ]:
# pip install plotly
# !pip install plotly


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

print('✅  Plotly imported')


In [ ]:
def compute_forecast_metrics(monthly, trend):
    """Compute RMSE, MAE, MAPE, R² for the in-sample OLS fit."""
    rev  = monthly['revenue'].values.astype(float)
    n    = len(rev)
    x    = np.arange(n, dtype=float)
    sl   = float(trend['slope_per_month'])
    ic   = float(trend['intercept_level'])
    pred = sl * x + ic
    err  = rev - pred
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae  = float(np.mean(np.abs(err)))
    nz   = rev != 0
    mape = float(np.mean(np.abs(err[nz] / rev[nz])) * 100) if nz.any() else 0.0
    ss_r = float(np.sum(err ** 2))
    ss_t = float(np.sum((rev - rev.mean()) ** 2))
    r2   = float(1 - ss_r / ss_t) if ss_t > 0 else 0.0
    return {'rmse': round(rmse,2), 'mae': round(mae,2),
            'mape': round(mape,2), 'r2': round(r2,4),
            'slope': round(sl,2),  'n': n}


def _ols_pi(x_hist, x_fut, residuals, sl, ic, z=1.28):
    """Proper OLS prediction interval: PI = ŷ ± z·s·sqrt(1+1/n+(x_f-x̄)²/SSx)"""
    n      = len(x_hist)
    s      = float(np.std(residuals, ddof=2)) if n > 2 else float(np.std(residuals))
    x_mean = float(x_hist.mean())
    ssx    = float(np.sum((x_hist - x_mean)**2))
    y_hat  = sl * x_fut + ic
    se     = np.sqrt(1 + 1/n + (x_fut - x_mean)**2 / max(ssx, 1e-10))
    margin = z * s * se
    return y_hat, np.maximum(y_hat - margin, 0.0), y_hat + margin


print('✅  Metric & PI functions defined')


In [ ]:
def plot_forecast_evaluation(engine_output, store_label='Store', dark_mode=True):
    """
    Full interactive Plotly forecast evaluation chart.
    
    Layers: daily bars, 7d/30d rolling ribbons, OLS trend,
    anomaly markers, 4-week forecast with OLS PI band,
    metric annotation box, monthly overview panel.
    """
    monthly    = engine_output['series']['monthly']
    daily      = engine_output['series']['daily']
    rolling    = engine_output['series']['rolling']
    trend      = engine_output['trend']
    anomalies  = engine_output['anomalies']
    forecast   = engine_output['forecast']
    metadata   = engine_output['metadata']
    seasonality= engine_output['seasonality']

    metrics = compute_forecast_metrics(monthly, trend)

    # Palette
    bg       = '#0f1117' if dark_mode else '#ffffff'
    paper_bg = '#161b22' if dark_mode else '#f8f9fa'
    txt_col  = '#e6edf3' if dark_mode else '#1a1a2e'
    grid_col = '#30363d' if dark_mode else '#e1e4e8'
    P = {
        'bar':    'rgba(99,179,237,0.35)' if dark_mode else 'rgba(59,130,246,0.25)',
        'line':   '#63b3ed' if dark_mode else '#2563eb',
        'trend':  '#f6ad55' if dark_mode else '#d97706',
        'r7':     '#68d391' if dark_mode else '#059669',
        'r7f':    'rgba(104,211,145,0.15)',
        'r30':    '#fc8181' if dark_mode else '#dc2626',
        'r30f':   'rgba(252,129,129,0.12)',
        'asp':    '#fc8181',
        'adp':    '#f6ad55',
        'fcl':    '#b794f4' if dark_mode else '#7c3aed',
        'fcf':    'rgba(183,148,244,0.15)',
    }

    # Trend line
    rev_h = monthly['revenue'].values.astype(float)
    x_h   = np.arange(len(rev_h), dtype=float)
    sl    = float(trend['slope_per_month'])
    ic    = float(trend['intercept_level'])
    trend_y = sl * x_h + ic

    # OLS PI for forecast
    resid = rev_h - trend_y
    wpm   = 4.33
    x_fut = np.array([len(x_h)-1 + (w/wpm) for w in range(1,5)])
    preds = forecast['predictions']
    fc_dates = pd.to_datetime([p['date'] for p in preds])
    fc_vals  = np.array([p['predicted_revenue'] for p in preds])
    _, fc_lo, fc_hi = _ols_pi(x_h, x_fut, resid, sl, ic)

    # Anomaly markers
    sp_x,sp_y,sp_t,dr_x,dr_y,dr_t = [],[],[],[],[],[]
    for a in anomalies['anomaly_details']:
        try: dt = pd.to_datetime(str(a['period']))
        except: continue
        lbl = (f"<b>{a['type'].upper()}</b> {a['period']}<br>"
               f"Actual: ${a['actual_revenue']:,.0f}<br>"
               f"Expected: ${a['expected_revenue']:,.0f}<br>"
               f"Z: {a['zscore']:+.2f}  |  {a['pct_deviation']:+.0f}%")
        if a['type']=='spike': sp_x.append(dt);sp_y.append(a['actual_revenue']);sp_t.append(lbl)
        else:                  dr_x.append(dt);dr_y.append(a['actual_revenue']);dr_t.append(lbl)

    d_arr = '▲' if trend['direction']=='increasing' else '▼' if trend['direction']=='decreasing' else '→'
    perf  = 'Excellent' if metrics['mape']<10 else 'Good' if metrics['mape']<20 else 'Moderate' if metrics['mape']<30 else 'Weak'

    title = (
        f"<b>DataSpeak — Forecast Evaluation | {store_label}</b><br>"
        f"<span style='font-size:13px;color:#8b949e'>"
        f"Period: {metadata['data_start']} → {metadata['data_end']} | "
        f"Trend: {d_arr} {trend['direction'].upper()} (slope ${sl:+,.0f}/mo, R²={metrics['r2']:.3f}) | "
        f"RMSE: ${metrics['rmse']:,.0f} | MAPE: {metrics['mape']:.1f}% ({perf})"
        f"</span>"
    )

    fig = make_subplots(rows=2,cols=1,row_heights=[0.72,0.28],shared_xaxes=True,
                        vertical_spacing=0.06,
                        subplot_titles=('Daily Revenue · Trend · Anomalies · Forecast',
                                        'Monthly Revenue Overview'))

    # Daily bars
    fig.add_trace(go.Bar(x=daily.index,y=daily['revenue'],name='Daily Revenue',
        marker_color=P['bar'],marker_line_width=0,
        hovertemplate='<b>%{x|%d %b %Y}</b><br>$%{y:,.0f}<extra></extra>'),row=1,col=1)

    # 7d rolling ribbon
    if 'rev_7d' in rolling.columns:
        fig.add_trace(go.Scatter(x=rolling.index,y=rolling['upper_7d'],line=dict(width=0),
            showlegend=False,hoverinfo='skip',fillcolor=P['r7f'],legendgroup='r7'),row=1,col=1)
        fig.add_trace(go.Scatter(x=rolling.index,y=rolling['lower_7d'],line=dict(width=0),
            fill='tonexty',fillcolor=P['r7f'],showlegend=False,hoverinfo='skip',legendgroup='r7'),row=1,col=1)
        fig.add_trace(go.Scatter(x=rolling.index,y=rolling['rev_7d'],name='7-Day Avg',
            line=dict(color=P['r7'],width=1.8),
            hovertemplate='7d avg: $%{y:,.0f}<extra></extra>',legendgroup='r7'),row=1,col=1)

    # 30d rolling ribbon
    if 'rev_30d' in rolling.columns:
        fig.add_trace(go.Scatter(x=rolling.index,y=rolling['upper_30d'],line=dict(width=0),
            showlegend=False,hoverinfo='skip',fillcolor=P['r30f'],legendgroup='r30'),row=1,col=1)
        fig.add_trace(go.Scatter(x=rolling.index,y=rolling['lower_30d'],line=dict(width=0),
            fill='tonexty',fillcolor=P['r30f'],showlegend=False,hoverinfo='skip',legendgroup='r30'),row=1,col=1)
        fig.add_trace(go.Scatter(x=rolling.index,y=rolling['rev_30d'],name='30-Day Avg',
            line=dict(color=P['r30'],width=1.8,dash='dot'),
            hovertemplate='30d avg: $%{y:,.0f}<extra></extra>',legendgroup='r30'),row=1,col=1)

    # OLS trend
    fig.add_trace(go.Scatter(x=monthly.index,y=trend_y,
        name=f'OLS Trend (${sl:+,.0f}/mo)',
        line=dict(color=P['trend'],width=2.5,dash='dash'),
        hovertemplate='Trend: $%{y:,.0f}<extra></extra>'),row=1,col=1)

    # Forecast CI band
    lhd = monthly.index[-1]; lhv = float(rev_h[-1])
    bx=[lhd]+list(fc_dates); by=[lhv]+list(fc_vals)
    blo=[lhv]+list(fc_lo);   bhi=[lhv]+list(fc_hi)
    fig.add_trace(go.Scatter(x=bx,y=bhi,line=dict(width=0),showlegend=False,
        hoverinfo='skip',fillcolor=P['fcf'],legendgroup='fc'),row=1,col=1)
    fig.add_trace(go.Scatter(x=bx,y=blo,line=dict(width=0),fill='tonexty',
        fillcolor=P['fcf'],showlegend=False,hoverinfo='skip',legendgroup='fc'),row=1,col=1)
    fig.add_trace(go.Scatter(x=bx,y=by,name='4-Week Forecast',
        line=dict(color=P['fcl'],width=2.5,dash='dashdot'),mode='lines+markers',
        marker=dict(size=8,symbol='diamond',color=P['fcl']),
        hovertemplate='Forecast: $%{y:,.0f}<extra></extra>',legendgroup='fc'),row=1,col=1)

    # Anomaly markers
    if sp_x:
        fig.add_trace(go.Scatter(x=sp_x,y=sp_y,mode='markers',name='Anomaly: Spike',
            marker=dict(color=P['asp'],size=14,symbol='triangle-up',
                        line=dict(color='white',width=1.5)),
            hovertemplate='%{text}<extra></extra>',text=sp_t),row=1,col=1)
    if dr_x:
        fig.add_trace(go.Scatter(x=dr_x,y=dr_y,mode='markers',name='Anomaly: Drop',
            marker=dict(color=P['adp'],size=14,symbol='triangle-down',
                        line=dict(color='white',width=1.5)),
            hovertemplate='%{text}<extra></extra>',text=dr_t),row=1,col=1)

    # Forecast start line
    fig.add_shape(type='line',xref='x',yref='paper',
        x0=lhd,x1=lhd,y0=0,y1=1,
        line=dict(color='rgba(150,150,150,0.5)',width=1.5,dash='dot'))
    fig.add_annotation(xref='x',yref='paper',x=lhd,y=0.99,
        text='◀ Forecast start',showarrow=False,
        font=dict(color='#8b949e',size=11),xanchor='right')

    # Monthly overview
    fig.add_trace(go.Bar(
        x=monthly.index,y=monthly['revenue'],name='Monthly',
        marker_color=[
            P['asp'] if str(monthly.index[i].to_period('M')) in anomalies['anomaly_periods']
            else P['line'] for i in range(len(monthly))
        ],showlegend=False,
        hovertemplate='<b>%{x|%b %Y}</b><br>$%{y:,.0f}<extra></extra>'),row=2,col=1)

    # Metric annotation
    sn = (f"✅ {seasonality['distinct_months']}mo" if seasonality['reliable']
          else f"⚠ {seasonality['distinct_months']}mo (low)")
    fig.add_annotation(
        xref='paper',yref='paper',x=1.0,y=0.97,xanchor='right',yanchor='top',
        text=(f"<b>Model Performance</b><br>"
              f"RMSE : ${metrics['rmse']:,.0f}<br>"
              f"MAPE : {metrics['mape']:.1f}%<br>"
              f"MAE  : ${metrics['mae']:,.0f}<br>"
              f"R²   : {metrics['r2']:.4f}<br>"
              f"Slope: ${metrics['slope']:+,.0f}/mo<br>"
              f"N obs: {metrics['n']} months<br>"
              f"Season: {sn}"),
        showarrow=False,
        bgcolor='rgba(30,40,55,0.85)' if dark_mode else 'rgba(255,255,255,0.90)',
        bordercolor='#30363d' if dark_mode else '#d1d5db',borderwidth=1,
        font=dict(family="'Courier New',monospace",size=11,color=txt_col),align='left')

    fig.update_layout(
        title=dict(text=title,x=0.0,xanchor='left',font=dict(size=15,color=txt_col)),
        plot_bgcolor=bg,paper_bgcolor=paper_bg,
        font=dict(family='Inter,Arial,sans-serif',color=txt_col),
        legend=dict(bgcolor='rgba(0,0,0,0)',bordercolor=grid_col,borderwidth=1,
                    font=dict(size=11),orientation='h',yanchor='bottom',
                    y=1.01,xanchor='left',x=0),
        hovermode='x unified',height=780,
        margin=dict(t=130,b=40,l=60,r=20),
        xaxis=dict(showgrid=True,gridcolor=grid_col,gridwidth=0.5,zeroline=False),
        xaxis2=dict(showgrid=True,gridcolor=grid_col,gridwidth=0.5,zeroline=False),
        yaxis=dict(title='Revenue ($)',showgrid=True,gridcolor=grid_col,
                   gridwidth=0.5,zeroline=False,tickformat='$,.0f'),
        yaxis2=dict(title='Monthly ($)',showgrid=True,gridcolor=grid_col,
                    gridwidth=0.5,zeroline=False,tickformat='$,.0f'),
        bargap=0.1)
    return fig


print('✅  plot_forecast_evaluation() defined')


### Run the Chart


In [ ]:
fig = plot_forecast_evaluation(
    report['engine_output'],
    store_label='Synthetic Demo Store',
    dark_mode=True,
)
fig.show()

# Save as standalone HTML (optional)
# fig.write_html('dataspeak_forecast_report.html', include_plotlyjs='cdn')
# print('Saved → dataspeak_forecast_report.html')


### Console Metric Summary


In [ ]:
def print_metric_summary(engine_output, store_label='Store'):
    monthly  = engine_output['series']['monthly']
    trend    = engine_output['trend']
    forecast = engine_output['forecast']
    m        = compute_forecast_metrics(monthly, trend)
    lbl      = ('✅ Good' if m['mape']<15 else '⚠  Moderate' if m['mape']<30 else '🔴 Weak')
    print(f"\n{'═'*58}")
    print(f"  FORECAST QUALITY METRICS — {store_label}")
    print(f"{'═'*58}")
    print(f"  RMSE         : ${m['rmse']:>12,.2f}")
    print(f"  MAE          : ${m['mae']:>12,.2f}")
    print(f"  MAPE         : {m['mape']:>12.2f}%   {lbl}")
    print(f"  R²           : {m['r2']:>12.4f}")
    print(f"  Slope/month  : ${m['slope']:>+11,.2f}")
    print(f"  Observations : {m['n']:>12d}  months")
    print(f"{'─'*58}")
    print(f"  Reliability  : {forecast['reliability'].upper()}")
    print(f"  Season corr  : {'Yes' if forecast['seasonality_applied'] else 'No'}")
    print(f"  ADF p-value  : {trend['adf_p_value']:.6f}")
    print(f"  Stationary   : {trend['is_stationary']}")
    print(f"{'═'*58}")
    print()
    print('  MAPE benchmark for retail:')
    print('    < 10%  → Excellent  (enterprise ERP)')
    print('    10-20% → Good       (acceptable for SME)')
    print('    20-30% → Moderate   (linear model at its limit)')
    print('    > 30%  → Weak       (Phase-2 upgrade recommended)')


print_metric_summary(report['engine_output'], store_label='Synthetic Demo Store')


---
# Advanced: Custom Threshold Configuration
Override only the knobs you want — the rest stay at defaults.


In [ ]:
# Example: stricter anomaly detection + higher R² bar for reliability
custom_thresholds = {
    'anomaly': {
        'base_zscore': 3.0,     # was 2.5
        'z_stable_cv': 2.5,     # was 2.0
    },
    'trend': {
        'reliable_r2': 0.70,    # was 0.60
    },
}

# Pass at construction — deep-merged with defaults
custom_agent = TimeSeriesAgent(
    groq_api_key=os.getenv('GROQ_API_KEY'),
    thresholds=custom_thresholds,
)

custom_report = custom_agent.run(
    raw_df=synthetic_df,
    filename='synthetic_strict.csv',
    generate_llm_summary=False,  # skip LLM for speed
)

print('Custom thresholds result:')
print(f"  Anomalies detected : {custom_report['engine_output']['anomalies']['total_detected']}")
print(f"  Trend reliability  : {custom_report['engine_output']['forecast']['reliability']}")


---
# Batch Processing: Multiple Stores
Reuse one agent instance across all stores.


In [ ]:
# Simulated multi-store batch (replace with real CSV paths)
import os

STORE_FILES = {
    'Store A': 'cleaned_data_1.csv',
    'Store B': 'cleaned_data_2.csv',
    # 'Store C': 'cleaned_data_3.csv',  # add more as needed
}

batch_agent   = TimeSeriesAgent(groq_api_key=os.getenv('GROQ_API_KEY'))
batch_results = {}

for store_name, csv_path in STORE_FILES.items():
    if not os.path.exists(csv_path):
        print(f'⚠  Skipping {store_name} — file not found: {csv_path}')
        continue
    print(f'\n Processing {store_name}...')
    report = process_csv(
        filepath=csv_path,
        agent=batch_agent,
        show_template_b=False,   # suppress deep-dive in batch mode
        show_template_c=True,
    )
    batch_results[store_name] = report

# Summary table
print(f"\n{'═'*60}")
print('BATCH SUMMARY')
print(f"{'─'*60}")
for name, r in batch_results.items():
    if r:
        t = r['engine_output']['trend']
        f = r['engine_output']['forecast']
        a = r['engine_output']['anomalies']
        print(f"{name:10} | trend: {t['direction']:12} | "
              f"R²={t['r_squared']:.3f} | anomalies: {a['total_detected']} | "
              f"30d forecast: ${f['next_month_estimate']:,.0f} ({f['reliability']})")
    else:
        print(f"{name:10} | FAILED")


---
# Summary

This notebook is the **complete DataSpeak Time Series & Forecasting Agent**, v3.0.

| Section | What it does |
|---------|-------------|
| **SchemaAdapter** | Fuzzy column matching, currency stripping, returns isolation |
| **TimeSeriesEngine** | OLS trend, ADF stationarity, Z-score anomalies, seasonal indices, 4-week forecast |
| **TemplateRenderer** | 3 business-language templates (executive, deep-dive, urgent alert) |
| **TimeSeriesAgent** | Pipeline orchestration + Groq/Llama-3.3 LLM integration |
| **Forecast Evaluator** | Plotly interactive chart + RMSE/MAE/MAPE/R² metrics |

### For Your Team
- **Farhat (Cleaning Agent)** → outputs go straight to `SchemaAdapter.standardize(df)`, any column names accepted
- **Motawea (Visualization Agent)** → use `report['engine_output']['series']` for chart-ready DataFrames

### Next Steps (Phase 2)
1. **Sprint 1** — Add holiday calendar flags (Ramadan, Eid) to seasonality
2. **Sprint 2** — Replace `_forecast()` with Prophet for series ≥ 12 months
3. **Sprint 3** — Add Holt-Winters fallback for series < 12 months
4. **Sprint 4** — XGBoost multivariate layer (requires Farhat's ad spend / sessions columns)
